# 面向长对话的贝叶斯自适应上下文选择 · Release研究任务工作本
**ARIN7101 · release-2.1 scaffold · 2026-11-05**

研究问题：在受控合成长对话中，利用问题特征的贝叶斯策略选择能否改善正确性与输入成本的取舍？

**这是任务脚手架，研究实现和结果评估由成员自行完成。** 这里只提供公共路径、实验设计草案、接口骨架与任务提示。第0节检查环境、建立统一目录并保存环境记录。各模块进度以相应单元与交接记录为准；初始化成功不代表任务完成。

先读[README](README.md)和[协作说明](docs/TEAM_WORKFLOW.md)。每个人须交付自己的代码、自己设计的验证、实际观察与局限和可追踪产物。参考完整本只作可行性背景，release不从中导入或执行实现；旧数值、图表与结论不属于本组已完成的研究。

| 成员 | 必须自己研究、编码的范围 | cell ID区间 | 接收人 |
|---|---|---|---|
| [M1](#member-m1) | 数据生成、公开信息边界、独立真值核查 | m1-* | M2/M4 |
| [M2](#member-m2) | 四策略、回答器、成本、评价及反例 | m2-* | M1/M3/M4 |
| [M3](#member-m3) | 特征、TS/CTS、后验数值验证、回放 | m3-* | M4 |
| [M4](#member-m4) | 开发选择、冻结、实验调度、统计与归档 | m4-* | M3复核/M5 |
| [M5](#member-m5) | 读取校验、可视化编码、结果解释与汇报 | m5-* | 全组 |


## 0 · 环境配置及路径约定
**本节代码已写好，全员首次下载仓库后直接按顺序运行本节两个代码单元。** 它负责依赖检查、公共导入、根目录定位、统一目录初始化、环境记录及安全保存工具。之后各成员使用这里的路径变量；无需自行拼接绝对路径。

### 0.1 · 首次准备与环境检查
建议Python 3.11或3.12。在仓库根目录创建虚拟环境并安装Jupyter（首次尚无法打开notebook时执行）：

    python -m venv .venv
    # Windows PowerShell：.\.venv\Scripts\Activate.ps1
    # macOS/Linux：source .venv/bin/activate
    python -m pip install -r requirements.txt
    python -m ipykernel install --user --name arin7101 --display-name "Python 3 (ARIN7101)"
    python -m jupyter lab

选择上述内核。下一单元使用当前内核的Python检查requirements；若依赖缺失或版本不符，默认自动安装，并要求重启内核后重跑，以免混用已加载的旧包。无需GPU、API密钥或外部数据。

PROJECT_ROOT默认自动从当前目录向上寻找仓库；若从仓库外启动内核，将它填为本机仓库根路径。不要把自己的绝对路径提交Git。AUTO_INSTALL_MISSING可改为False，只检查并给出安装提示。


In [ ]:
# 0.1：全员直接运行。只处理环境与公共导入。
import sys
import os
import re
import json
import platform
import subprocess
from pathlib import Path
from datetime import datetime
from importlib import metadata
from packaging.requirements import Requirement
from IPython.display import display

PROJECT_ROOT = None  # 通常无需填写；仓库外启动时可填本机路径
AUTO_INSTALL_MISSING = True

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("本项目统一使用Python 3.11或3.12；请切换对应Jupyter内核。")
_candidates = ([Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT
               else [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in _candidates
             if (p / "ARIN7101_Context_Bandit_Release_Notebook.ipynb").is_file()
             and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("未定位仓库，请填写PROJECT_ROOT或从仓库内启动Jupyter。")

_requirements = []
for _line in (ROOT / "requirements.txt").read_text(encoding="utf-8-sig").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if _line:
        _req = Requirement(_line)
        if _req.marker is None or _req.marker.evaluate():
            _requirements.append(_req)

_problems = []
for _req in _requirements:
    try:
        _version = metadata.version(_req.name)
    except metadata.PackageNotFoundError:
        _problems.append(f"{_req.name} 未安装")
    else:
        if _version not in _req.specifier:
            _problems.append(f"{_req.name}={_version}，要求 {_req.specifier}")
if _problems:
    print("需要处理的依赖：\n- " + "\n- ".join(_problems))
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError(f'请用当前内核Python安装依赖："{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"')
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
                   check=True)
    raise RuntimeError("依赖已安装。请Restart Kernel，重新运行第0节，避免混用已加载的旧包。")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "executable": sys.executable,
    "packages": {_req.name: metadata.version(_req.name) for _req in _requirements},
}
print("环境检查通过")
print("项目根目录：", ROOT)
print("当前内核：", sys.executable)
display(pd.DataFrame(list(ENVIRONMENT["packages"].items()), columns=["package", "version"]))


### 0.2 · 初始化工作目录和公共路径
运行下一单元，默认建立本机练习目录outputs/development/local_setup及其六个子目录，并写入config/environment.json。已有文件与人工记录不覆盖。重复执行同一个development/demo配置可复用目录；正式run必须是新目录。

**每次实际实验开始前修改MODE和RUN_ID**，例如development / m1_trial01，或demo / team_integration01。成员交接必须明确run路径；同名run只有在配置、输入与来源一致时才能合用。正式运行由M4完成审查与冻结后初始化，初始化不等于获得正式执行资格。

| 路径/变量 | 用途与负责成员 | 本节行为 | Git |
|---|---|---|---|
| ROOT | 仓库根目录，自动定位 | 查找，不改工作目录 | — |
| TEAM_DIR = team_workspace/ | 姓名、进度、接口决策、正式尝试登记；全员/M4 | 确保目录存在，不改人工记录 | 提交 |
| DELIVERABLES_DIR = deliverables/ | 个人研究说明、最终slides、提交清单；全员 | 确保目录存在 | 提交小文件，视频用链接 |
| DOCS_DIR / REFERENCE_DIR | 协作规范、原计划、历史参考 | 仅定位/确保目录存在 | 提交 |
| OUTPUTS_DIR = outputs/ | 自动产物总目录 | 建立development/demo/formal三个分区 | 忽略 |
| RUN_DIR = outputs/MODE/RUN_ID/ | 本次开发/实验的唯一根目录 | 初始化；formal禁止复用已有目录 | 忽略 |
| DATA_DIR = RUN_DIR/data/ | 生成对话与原始数据；M1/M4 | 创建空目录 | 忽略 |
| TABLES_DIR = RUN_DIR/tables/ | 开发评价、回放、统计表、绘图数据；M2–M5 | 创建空目录 | 忽略 |
| AUDIT_DIR = RUN_DIR/audit/ | 真值/策略/方法检查、十题审查与证据副本；M1–M4 | 创建空目录 | 忽略 |
| CONFIG_DIR = RUN_DIR/config/ | 环境记录、冻结配置、代码快照；公共设施/M4 | 写环境记录，其余由成员完成 | 忽略 |
| FIGURES_DIR = RUN_DIR/figures/ | 自己编写代码生成的PNG/SVG；M5 | 创建空目录 | 忽略 |
| REPORTS_DIR = RUN_DIR/reports/ | 结果交接、逐图解读、图表清单；M4/M5 | 创建空目录 | 忽略 |
| RUN_DIR/manifest.json | 整个实验的来源与文件校验清单；M4 | 不生成，留给M4实现 | 忽略 |

**统一使用方式：**
- 自己实现的函数接收RUN_DIR；单文件用artifact_path(MODE, RUN_ID, area, filename)取路径。
- 已提供save_json(path, value)和save_csv(path, dataframe)；它们限定outputs内部、UTF-8编码，拒绝覆盖已有文件。不再让每人重复写路径/通用保存代码。
- 绘图、压缩对话等专用导出仍由成员实现，使用上述目录。JSON、CSV的内容与研究指标仍由成员计算。
- M5读取M4交付的明确路径作为SOURCE_RUN，不重新初始化/改写上游formal目录；可以另用自己的development run试绘。
- 本节只有环境信息，不生成研究配置冻结、数据、统计、图或成员签名。使用新RUN_ID保留修改前产物；不要以重跑初始化来清理旧结果。


In [ ]:
# 0.2：全员直接运行；做实际实验时改这两个值，不改下面的统一目录规则。
MODE = "development"     # development / demo / formal
RUN_ID = "local_setup"   # 正式实验由M4在冻结后使用全新ID

TEAM_DIR = ROOT / "team_workspace"
DELIVERABLES_DIR = ROOT / "deliverables"
DOCS_DIR = ROOT / "docs"
REFERENCE_DIR = ROOT / "reference"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUT_AREAS = ("data", "tables", "audit", "config", "figures", "reports")
MODES = ("development", "demo", "formal")

def _safe_component(value):
    if (not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value)
            or value.endswith(".") or value.split(".", 1)[0].upper() in
            {"CON", "PRN", "AUX", "NUL", *[f"COM{i}" for i in range(1, 10)],
             *[f"LPT{i}" for i in range(1, 10)]}):
        raise ValueError("名称须以字母/数字开头，只含字母、数字、下划线、连字符和点；不可用系统保留名。")
    return value

def _run_path(mode, run_id):
    if mode not in MODES:
        raise ValueError(f"mode须为{MODES}")
    path = OUTPUTS_DIR / mode / _safe_component(run_id)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def artifact_path(mode, run_id, area, filename):
    """返回统一文件路径；先使用init_run初始化该run，不在此处生成研究结果。"""
    if area not in OUTPUT_AREAS:
        raise ValueError(f"area须为{OUTPUT_AREAS}")
    path = _run_path(mode, run_id) / area / _safe_component(filename)
    if not path.resolve().is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("路径必须位于outputs内。")
    return path

def _writable_output(path):
    path = Path(path).resolve()
    if not path.is_relative_to(OUTPUTS_DIR.resolve()):
        raise ValueError("通用保存工具只写outputs；人工记录请本人编辑。")
    if not path.parent.is_dir():
        raise FileNotFoundError("输出目录不存在；请先init_run。")
    return path

def save_json(path, value):
    """统一UTF-8/严格JSON，拒绝覆盖；不代替研究计算或签名。"""
    payload = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
    with _writable_output(path).open("x", encoding="utf-8") as stream:
        stream.write(payload)

def save_csv(path, dataframe):
    """保存成员计算得到的DataFrame；统一UTF-8、无索引、拒绝覆盖。"""
    dataframe.to_csv(_writable_output(path), index=False, encoding="utf-8", mode="x")

def init_run(mode, run_id):
    """创建六个目录和环境记录；开发/demo可复用，formal禁止复用。"""
    run_dir = _run_path(mode, run_id)
    if mode == "formal" and run_dir.exists():
        raise FileExistsError("正式run目录已存在；请保留原目录，不能重跑初始化覆盖。")
    # 先检查既有环境，不改已有记录；环境变化应使用新的run_id。
    env_path = run_dir / "config" / "environment.json"
    if env_path.exists():
        existing = json.loads(env_path.read_text(encoding="utf-8"))
        if (existing.get("python"), existing.get("packages")) != (
                ENVIRONMENT["python"], ENVIRONMENT["packages"]):
            raise RuntimeError("本run已记录不同Python/依赖版本；请用新RUN_ID保留环境来源。")
    run_dir.mkdir(parents=True, exist_ok=(mode != "formal"))
    for area in OUTPUT_AREAS:
        directory = run_dir / area
        if not directory.resolve().is_relative_to(run_dir.resolve()):
            raise ValueError("子目录不能指向run以外的位置。")
        directory.mkdir(exist_ok=True)
    if not env_path.exists():
        save_json(env_path, {
            **ENVIRONMENT, "mode": mode, "run_id": run_id,
            "created_at": datetime.now().astimezone().isoformat(),
            "status": "environment_initialized_only",
        })
    return run_dir

for directory in (TEAM_DIR, DELIVERABLES_DIR, DOCS_DIR, REFERENCE_DIR, OUTPUTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for mode in MODES:
    (OUTPUTS_DIR / mode).mkdir(exist_ok=True)

RUN_DIR = init_run(MODE, RUN_ID)
RUN_PATHS = {area: RUN_DIR / area for area in OUTPUT_AREAS}
DATA_DIR = RUN_PATHS["data"]
TABLES_DIR = RUN_PATHS["tables"]
AUDIT_DIR = RUN_PATHS["audit"]
CONFIG_DIR = RUN_PATHS["config"]
FIGURES_DIR = RUN_PATHS["figures"]
REPORTS_DIR = RUN_PATHS["reports"]

print("初始化完成；未生成研究结果。")
print("RUN_DIR =", RUN_DIR)
display(pd.DataFrame(
    [{"area": area, "relative_path": directory.relative_to(ROOT).as_posix(),
      "absolute_path": str(directory)} for area, directory in RUN_PATHS.items()]
))
print("环境记录：", CONFIG_DIR / "environment.json")

# 在成员完成计算后使用（这里只示范路径，不生成假的结果文件）：
# save_csv(artifact_path(MODE, RUN_ID, "tables", "development_outcomes.csv"), outcomes)
# save_json(artifact_path(MODE, RUN_ID, "audit", "truth_checks.json"), checks)
# M5使用明确的SOURCE_RUN读取M4交付物，不调用init_run去重建上游formal目录。


## 共同研究设计与成员探索空间
以下设计草案供M1–M5研究；与第0节的环境初始化分开，确认冻结由M4负责。

**原计划提供的设计起点，开发结束前由全组确认冻结：**
- 6个项目、4种属性，英文模板历史含focus、事实更新、旧版本与干扰；每对话300题。
- stable的近期题概率0.9；abrupt在第151题起从0.9变0.1。改变的是提问需求分布。
- recent_short：最近最多3条，预算128；recent_long：连续近期窗口，512；tfidf：原问题检索最多8个候选，256；resolved_tfidf：用公开历史消解指代/同义词后检索，256。
- U = Y − λC/Cref，λ=0.5、Cref=512。C是整个输入的正则长度代理，不是真实token费用；预算仅限制所选上下文。
- 开发每场景3条、demo每场景6条、正式每场景30条。TS/CTS算法seed为17/37，每对话重置。
- 主比较：CTS与各场景开发集选定的固定臂；TS为次比较；RuleRouter为模板规则参照。算法seed先在对话内平均，再按独立对话配对bootstrap，报告95%和两主比较校正97.5%区间。

**成员需要自行决定并研究：** 具体模板与版本生成细节、检索平局、数值优化、失败处理、测试案例与图型选择。记录理由、比较和局限，不能只复制完整本。对共同设计的修改须在正式评估前确认；测试结果之后的修改另标探索/修正。

**并行起步：** M1先做可追踪小历史；M2用自建历史；M3用自建四臂小表；M4设计统计手算案例；M5用明确标为fixture的虚构表研究绘图。示意数据不进入真实结果。

**接通顺序：** M1生成器 → M2评价器 → M1/M2共同审计 → M3回放 → M4集成与冻结 → 正式运行 → M5解释。未实现函数被调用时明确报错，不用空表/固定返回值掩盖未完成。


In [ ]:
ARMS = ("recent_short", "recent_long", "tfidf", "resolved_tfidf")
SCENARIOS = ("stable", "abrupt")
METHODS = ARMS + ("TS", "LaplaceCTS", "RuleRouter")
PROTOCOL_DRAFT = {
    "T": 300, "n_dev_per_scenario": 3, "n_demo_per_scenario": 6,
    "n_formal_per_scenario": 30, "dev_seed": 130000, "demo_seed": 730000,
    "formal_seed": None,  # M4与全组确认未使用的seed后填写
    "algorithm_seeds": [17, 37], "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9, "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25, "initial_conflict_fraction": 0.25,
    "budgets": dict(zip(ARMS, [128, 512, 256, 256])),
    "lambda": 0.5, "cost_reference": 512, "refit_every": 5,
    "prior_precision": 1.0, "bootstrap_seed": 2026, "n_bootstrap": 2000,
}


### 共同接口
详细字段见[输出契约](docs/OUTPUT_CONTRACT.md)。骨架参数名是交接约定；模块内部实现由成员选择。
- episode：seed/scenario/history/probes；message：id/text；probe：t/question/end/gold/evidence/old_values/qtype/state。
- public_view仅含question/history；gold、隐藏题型、未选反馈只供评估器访问。
- 四臂表：seed,scenario,t,arm,question,y,cost,utility,coverage,outdated,construction_ms,qtype,state。
- 回放表：seed,scenario,t,method,alg_seed,arm,y,cost,utility,coverage,outdated。
- 对话指标：scenario,seed,method,y,cost,utility,coverage,outdated。
- 主效应表：scenario,method,baseline,n,effect,low,high,family_low,family_high。

消费者自行检查字段、唯一键、规模与来源。上游未完成时自行构造明确标为fixture的最小输入，不用完整参考结果填空。


<a id="member-m1"></a>
## M1 · 数据与问题定义
**你要编写生成器、公开视图和独立核查器，研究数据机制。任务区间m1-*。**
输入：共同设计；输出：episode、数据字典、开发数据、核查证据；M2/M4接收。

1. **先设计。** 用纸面例子定义focus/its、对象、属性、最新事实和过期值；决定多版本、复述、同义词如何生成，记录选择理由。
2. **先小后大编码。** 从少量题目的生成器扩展到两个场景与300题；保持当前题只能看历史前缀。
3. **独立核查。** 设计另一条证据追踪路径，不把生成器存的gold读出来就当证明；写复现、指代、更新、旧值、切换边界案例。
4. **交付与联调。** 先给M2微型episode和字典，再导出开发集。M2导出四臂审计后，你追踪10题真值，M2交叉复核。
5. **诊断与解释。** 检查实际题型/版本分布，说明是否人为偏向某种路由；发现问题在开发阶段修改，不预设策略一定胜出。

**验收方向（案例由你设计）：** seed可复现、id连续、无未来证据、真值可追溯、切换正确。接收人能从一条样例找到答案依据。
**输出：** data/development_dialogues.jsonl.gz；audit/data_dictionary.json、truth_checks.json；tables/development_data_profile.csv。使用明确的development run路径。


In [ ]:
import gzip as _m1_gzip
import json as _m1_json
import math as _m1_math
import re as _m1_re
from pathlib import Path as _m1_Path

M1_ATTRIBUTES = ("dataset", "language", "metric", "deadline")
M1_ATTRIBUTE_ALIASES = {
    "dataset": "data source",
    "language": "coding language",
    "metric": "evaluation measure",
    "deadline": "due date",
}
M1_SYSTEM_CONTEXT = (
    "Please use this setting in subsequent work and keep other project requirements unchanged."
)
M1_DEFAULT_CONFIG = {
    "T": 300,
    "n_projects": 6,
    "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9,
    "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25,
    "initial_conflict_fraction": 0.25,
    "alias_probability": 0.5,
}
if "PROTOCOL_DRAFT" in globals():
    M1_DEFAULT_CONFIG.update(PROTOCOL_DRAFT)

_M1_ATTR_PATTERN = "|".join(
    _m1_re.escape(term)
    for term in sorted(
        [*M1_ATTRIBUTES, *M1_ATTRIBUTE_ALIASES.values()],
        key=len,
        reverse=True,
    )
)
_M1_FACT_RE = _m1_re.compile(
    r"^For project (?P<project>P\d+): the (?P<attribute>"
    + _M1_ATTR_PATTERN
    + r") is (?P<value>V\d+)\."
)
_M1_FOCUS_RE = _m1_re.compile(r"^Focus is project (?P<project>P\d+)\.")
_M1_PROJECT_RE = _m1_re.compile(r"\bP\d+\b")
_M1_ALIAS_TO_ATTRIBUTE = {
    term: attribute
    for attribute, alias in M1_ATTRIBUTE_ALIASES.items()
    for term in (attribute, alias)
}


def _m1_config(cfg):
    settings = dict(M1_DEFAULT_CONFIG)
    if cfg is not None:
        settings.update(dict(cfg))
    total = settings["T"]
    projects = settings["n_projects"]
    if isinstance(total, bool) or not isinstance(total, int) or total < 1:
        raise ValueError("cfg['T'] must be a positive integer")
    if isinstance(projects, bool) or not isinstance(projects, int) or projects < 2:
        raise ValueError("cfg['n_projects'] must be an integer of at least 2")
    for name in (
        "p_recent_stable",
        "p_recent_abrupt_before",
        "p_recent_abrupt_after",
        "update_probability",
        "initial_conflict_fraction",
        "alias_probability",
    ):
        value = settings[name]
        if not isinstance(value, (int, float)) or not _m1_math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"cfg[{name!r}] must be a finite probability in [0, 1]")
        settings[name] = float(value)
    return settings


def _m1_canonical_attribute(label):
    return _M1_ALIAS_TO_ATTRIBUTE.get(label.strip().lower())


def _m1_parse_fact(text):
    match = _M1_FACT_RE.match(text)
    if not match:
        return None
    result = match.groupdict()
    result["attribute"] = _m1_canonical_attribute(result["attribute"])
    return result


def _m1_parse_question(question, focus):
    project_match = _M1_PROJECT_RE.search(question)
    project = project_match.group() if project_match else focus
    attribute = None
    for term in sorted(_M1_ALIAS_TO_ATTRIBUTE, key=len, reverse=True):
        if _m1_re.search(r"\b" + _m1_re.escape(term) + r"\b", question, _m1_re.IGNORECASE):
            attribute = _M1_ALIAS_TO_ATTRIBUTE[term]
            break
    return project, attribute, project_match is not None


def generate_episode(seed, scenario, cfg):
    """Generate one reproducible synthetic dialogue using only its local RNG."""
    if isinstance(seed, bool) or not isinstance(seed, (int, np.integer)) or int(seed) < 0:
        raise ValueError("seed must be a non-negative integer")
    seed = int(seed)
    if scenario not in ("stable", "abrupt"):
        raise ValueError("scenario must be 'stable' or 'abrupt'")
    settings = _m1_config(cfg)
    rng = np.random.default_rng(seed)
    projects = [f"P{i + 1}" for i in range(settings["n_projects"])]
    history = []
    probes = []
    facts = {}
    serial = 0

    def append(text):
        message_id = len(history)
        history.append({"id": message_id, "text": text})
        return message_id

    def write_fact(project, attribute, *, update, use_alias):
        nonlocal serial
        key = (project, attribute)
        previous = facts.get(key)
        if previous is None or update:
            serial += 1
            value = f"V{seed * 1_000_000 + serial}"
            versions = [] if previous is None else list(previous["versions"])
            if not versions or versions[-1] != value:
                versions.append(value)
        else:
            value = previous["value"]
            versions = list(previous["versions"])
        label = M1_ATTRIBUTE_ALIASES[attribute] if use_alias else attribute
        evidence = append(
            f"For project {project}: the {label} is {value}. {M1_SYSTEM_CONTEXT}"
        )
        facts[key] = {"value": value, "versions": versions, "evidence": evidence}

    # Seed every project-attribute pair so historical questions always have evidence.
    for project in projects:
        for attribute in M1_ATTRIBUTES:
            conflict = rng.random() < settings["initial_conflict_fraction"]
            initial_count = 2 if conflict else 1
            for version_index in range(initial_count):
                write_fact(
                    project,
                    attribute,
                    update=True,
                    use_alias=(version_index == initial_count - 1)
                    and (rng.random() < settings["alias_probability"]),
                )

    for t in range(settings["T"]):
        project = str(rng.choice(projects))
        attribute = str(rng.choice(M1_ATTRIBUTES))
        append(f"Focus is project {project}. We are reviewing its latest requirements.")
        is_update = rng.random() < settings["update_probability"]
        use_fact_alias = rng.random() < settings["alias_probability"]
        write_fact(project, attribute, update=is_update, use_alias=use_fact_alias)
        append("The next meeting will review progress and remaining questions.")

        if scenario == "stable":
            state = "A"
            p_recent = settings["p_recent_stable"]
        else:
            state = "A" if t < settings["T"] // 2 else "B"
            p_recent = (
                settings["p_recent_abrupt_before"]
                if state == "A"
                else settings["p_recent_abrupt_after"]
            )
        is_recent = rng.random() < p_recent
        if is_recent:
            target_project, target_attribute = project, attribute
            reference = "its"
            qtype = "recent"
        else:
            target_project = str(rng.choice([p for p in projects if p != project]))
            target_attribute = str(rng.choice(M1_ATTRIBUTES))
            reference = target_project
            qtype = "historical"
        question_term = (
            M1_ATTRIBUTE_ALIASES[target_attribute]
            if rng.random() < settings["alias_probability"]
            else target_attribute
        )
        probe_question = f"What is {reference} current {question_term}?"
        target_fact = facts[(target_project, target_attribute)]
        probes.append(
            {
                "t": t,
                "question": probe_question,
                "end": len(history),
                "gold": target_fact["value"],
                "evidence": target_fact["evidence"],
                "old_values": list(target_fact["versions"][:-1]),
                "qtype": qtype,
                "state": state,
            }
        )

    return {
        "seed": seed,
        "scenario": scenario,
        "history": history,
        "probes": probes,
    }


def public_view(episode, probe):
    """Return exactly the current question and its public history prefix."""
    if not isinstance(episode, dict) or not isinstance(probe, dict):
        raise TypeError("episode and probe must be dictionaries")
    end = probe.get("end")
    history = episode.get("history")
    if isinstance(end, bool) or not isinstance(end, int) or not isinstance(history, list):
        raise ValueError("probe.end must be an integer and episode.history a list")
    if not 0 <= end <= len(history):
        raise ValueError("probe.end is outside the episode history")
    return {
        "question": probe["question"],
        "history": [{"id": message["id"], "text": message["text"]} for message in history[:end]],
    }


def audit_truth(episode):
    """Independently reconstruct each answer from parsed public text, not stored gold."""
    episode_errors = []
    history = episode.get("history") if isinstance(episode, dict) else None
    probes = episode.get("probes") if isinstance(episode, dict) else None
    seed = episode.get("seed") if isinstance(episode, dict) else None
    scenario = episode.get("scenario") if isinstance(episode, dict) else None
    if not isinstance(history, list) or not isinstance(probes, list):
        return {
            "seed": seed,
            "scenario": scenario,
            "passed": False,
            "question_count": 0,
            "errors": ["episode must contain history and probes lists"],
            "questions": [],
        }
    expected_ids = list(range(len(history)))
    actual_ids = [message.get("id") if isinstance(message, dict) else None for message in history]
    if actual_ids != expected_ids:
        episode_errors.append("history message IDs are not continuous from zero")
    actual_t = [probe.get("t") if isinstance(probe, dict) else None for probe in probes]
    if actual_t != list(range(len(probes))):
        episode_errors.append("probe t values are not continuous from zero")

    question_checks = []
    for probe in probes:
        errors = []
        t = probe.get("t")
        end = probe.get("end")
        if isinstance(end, bool) or not isinstance(end, int) or not 0 <= end <= len(history):
            question_checks.append(
                {"seed": seed, "scenario": scenario, "t": t, "passed": False,
                 "errors": ["end is outside the available history"]}
            )
            continue
        prefix = history[:end]
        latest_focus = None
        values = {}
        evidence = {}
        versions = {}
        for message in prefix:
            if not isinstance(message, dict) or not isinstance(message.get("text"), str):
                errors.append("visible history contains a malformed message")
                continue
            focus_match = _M1_FOCUS_RE.match(message["text"])
            if focus_match:
                latest_focus = focus_match.group("project")
            parsed = _m1_parse_fact(message["text"])
            if parsed is None:
                continue
            key = (parsed["project"], parsed["attribute"])
            value = parsed["value"]
            values[key] = value
            evidence[key] = message.get("id")
            observed = versions.setdefault(key, [])
            if not observed or observed[-1] != value:
                observed.append(value)

        question = probe.get("question", "")
        if not isinstance(question, str):
            errors.append("question is not text")
            question = ""
        target_project, target_attribute, explicit_project = _m1_parse_question(question, latest_focus)
        if target_project is None:
            errors.append("could not resolve the question project from public history")
        if target_attribute is None:
            errors.append("could not resolve the question attribute")
        key = (target_project, target_attribute)
        derived_gold = values.get(key)
        derived_evidence = evidence.get(key)
        derived_old_values = list(versions.get(key, [])[:-1]) if derived_gold is not None else []
        if derived_gold is None:
            errors.append("no matching fact exists in the visible history prefix")
        if probe.get("gold") != derived_gold:
            errors.append("stored gold does not match the independently parsed latest value")
        if probe.get("evidence") != derived_evidence:
            errors.append("stored evidence does not point to the latest parsed fact message")
        if probe.get("old_values") != derived_old_values:
            errors.append("stored old_values does not match the parsed prior versions")
        stored_evidence = probe.get("evidence")
        if (isinstance(stored_evidence, bool) or not isinstance(stored_evidence, int)
                or not 0 <= stored_evidence < end):
            errors.append("stored evidence is not strictly inside the visible history prefix")
        expected_qtype = "historical" if explicit_project else "recent"
        if target_project is not None and latest_focus is not None:
            expected_qtype = "recent" if target_project == latest_focus else "historical"
        if probe.get("qtype") != expected_qtype:
            errors.append("qtype does not agree with the resolved project and current focus")
        if scenario == "stable":
            expected_state = "A"
        elif scenario == "abrupt":
            expected_state = "A" if isinstance(t, int) and t < len(probes) // 2 else "B"
        else:
            expected_state = None
            errors.append("scenario must be stable or abrupt")
        if expected_state is not None and probe.get("state") != expected_state:
            errors.append("state does not agree with the scenario switch boundary")
        visible = {"question", "history"}
        if set(public_view(episode, probe)) != visible:
            errors.append("public_view exposes fields outside question/history")
        question_checks.append(
            {
                "seed": seed,
                "scenario": scenario,
                "t": t,
                "target_project": target_project,
                "target_attribute": target_attribute,
                "latest_focus": latest_focus,
                "qtype": probe.get("qtype"),
                "state": probe.get("state"),
                "derived_gold": derived_gold,
                "stored_gold": probe.get("gold"),
                "derived_evidence": derived_evidence,
                "stored_evidence": probe.get("evidence"),
                "derived_old_values": derived_old_values,
                "stored_old_values": probe.get("old_values"),
                "history_end": end,
                "passed": not errors,
                "errors": errors,
            }
        )
    passed = not episode_errors and all(row["passed"] for row in question_checks)
    return {
        "seed": seed,
        "scenario": scenario,
        "passed": passed,
        "question_count": len(probes),
        "errors": episode_errors,
        "questions": question_checks,
    }


def _m1_profile(episodes, audit_records):
    rows = []
    for episode, audit in zip(episodes, audit_records):
        probe_checks = {row["t"]: row for row in audit["questions"]}
        history = episode["history"]
        for probe in episode["probes"]:
            check = probe_checks[probe["t"]]
            focus_ids = [
                message["id"] for message in history[:probe["end"]]
                if _M1_FOCUS_RE.match(message["text"])
            ]
            focus_fact = None
            if focus_ids:
                fact_index = focus_ids[-1] + 1
                if fact_index < probe["end"]:
                    focus_fact = _m1_parse_fact(history[fact_index]["text"])
            previous_values = []
            if focus_fact:
                focus_key = (focus_fact["project"], focus_fact["attribute"])
                for message in history[: focus_ids[-1] + 1]:
                    parsed = _m1_parse_fact(message["text"])
                    if parsed and (parsed["project"], parsed["attribute"]) == focus_key:
                        if not previous_values or previous_values[-1] != parsed["value"]:
                            previous_values.append(parsed["value"])
            rows.append(
                {
                    "seed": episode["seed"],
                    "scenario": episode["scenario"],
                    "t": probe["t"],
                    "state": probe["state"],
                    "qtype": probe["qtype"],
                    "has_old_values": bool(probe["old_values"]),
                    "n_old_values": len(probe["old_values"]),
                    "history_messages": probe["end"],
                    "evidence_lag_messages": probe["end"] - 1 - probe["evidence"],
                    "focus_fact_updated": bool(
                        focus_fact and previous_values and previous_values[-1] != focus_fact["value"]
                    ),
                    "focus_fact_uses_alias": bool(
                        focus_fact
                        and focus_fact["attribute"]
                        and history[focus_ids[-1] + 1]["text"].split(": the ", 1)[1].split(" is ", 1)[0]
                        == M1_ATTRIBUTE_ALIASES[focus_fact["attribute"]]
                    ),
                    "target_project": check["target_project"],
                    "target_attribute": check["target_attribute"],
                }
            )
    detail = pd.DataFrame(rows)
    profile = (
        detail.groupby(["scenario", "state"], as_index=False)
        .agg(
            dialogues=("seed", "nunique"),
            questions=("t", "size"),
            recent_fraction=("qtype", lambda s: float((s == "recent").mean())),
            historical_fraction=("qtype", lambda s: float((s == "historical").mean())),
            questions_with_old_values=("has_old_values", "sum"),
            old_value_fraction=("has_old_values", "mean"),
            mean_old_value_count=("n_old_values", "mean"),
            mean_history_messages=("history_messages", "mean"),
            mean_evidence_lag_messages=("evidence_lag_messages", "mean"),
            focus_update_fraction=("focus_fact_updated", "mean"),
            focus_fact_alias_fraction=("focus_fact_uses_alias", "mean"),
            distinct_target_projects=("target_project", "nunique"),
            distinct_target_attributes=("target_attribute", "nunique"),
        )
        .sort_values(["scenario", "state"])
        .reset_index(drop=True)
    )
    return detail, profile


def export_development_data(episodes, run_dir):
    """Export audited development episodes, field dictionary, checks, and profile."""
    run_dir = _m1_Path(run_dir).resolve()
    if "OUTPUTS_DIR" not in globals() or not run_dir.is_relative_to(_m1_Path(OUTPUTS_DIR).resolve()):
        raise ValueError("M1 exports must be written inside outputs/<mode>/<run_id>")
    required_areas = ("data", "tables", "audit")
    if not run_dir.is_dir() or any(not (run_dir / area).is_dir() for area in required_areas):
        raise FileNotFoundError("Initialize the run and its data/tables/audit directories first")
    episodes = list(episodes)
    if not episodes:
        raise ValueError("episodes must not be empty")
    keys = [(ep.get("scenario"), ep.get("seed")) for ep in episodes]
    if len(set(keys)) != len(keys):
        raise ValueError("episode scenario/seed pairs must be unique")
    audit_records = [audit_truth(episode) for episode in episodes]
    if not all(record["passed"] for record in audit_records):
        raise ValueError("Refusing to export episodes that fail independent truth checks")
    detail, profile = _m1_profile(episodes, audit_records)

    data_path = run_dir / "data" / "development_dialogues.jsonl.gz"
    dictionary_path = run_dir / "audit" / "data_dictionary.json"
    checks_path = run_dir / "audit" / "truth_checks.json"
    profile_path = run_dir / "tables" / "development_data_profile.csv"
    targets = (data_path, dictionary_path, checks_path, profile_path)
    existing = [str(path) for path in targets if path.exists()]
    if existing:
        raise FileExistsError("M1 refuses to overwrite existing artifacts: " + ", ".join(existing))

    with _m1_gzip.open(data_path, "xt", encoding="utf-8", newline="\n") as stream:
        for episode in episodes:
            stream.write(
                _m1_json.dumps(episode, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
            )
    save_json(
        dictionary_path,
        {
            "schema_version": "m1-1.0",
            "generator": "ARIN7101 M1 reproducible synthetic template generator",
            "episode_fields": {
                "seed": "Non-negative integer RNG seed for this independent dialogue.",
                "scenario": "stable or abrupt query-demand scenario.",
                "history": "Ordered public messages; message IDs start at zero and are continuous.",
                "probes": "Ordered query records; each end is an exclusive history prefix boundary.",
            },
            "message_fields": {
                "id": "Zero-based message index within the episode.",
                "text": "English synthetic focus, fact, or distractor message.",
            },
            "probe_fields": {
                "t": "Zero-based query index.",
                "question": "Question visible to a strategy.",
                "end": "Exclusive upper bound; public history is history[:end].",
                "gold": "Current value reconstructed from all facts before this query.",
                "evidence": "Message ID of the last supporting statement before this query.",
                "old_values": "Distinct earlier values for the queried project-attribute pair.",
                "qtype": "recent when the question refers to the current focus; historical otherwise.",
                "state": "A/B demand state; only abrupt changes from A to B at T//2.",
            },
            "public_strategy_fields": ["question", "history"],
            "evaluator_only_fields": ["gold", "evidence", "old_values", "qtype", "state", "scenario"],
            "attributes_and_aliases": M1_ATTRIBUTE_ALIASES,
            "message_templates": {
                "focus": "Focus is project Pn. We are reviewing its latest requirements.",
                "fact": "For project Pn: the <attribute or alias> is Vn. <shared system context>",
                "distractor": "The next meeting will review progress and remaining questions.",
            },
            "generation_rules": {
                "projects_per_episode": len(
                    {
                        parsed["project"]
                        for message in episodes[0]["history"]
                        if (parsed := _m1_parse_fact(message["text"])) is not None
                    }
                ),
                "questions_per_episode": sorted({len(episode["probes"]) for episode in episodes}),
                "scenario_switch_index": "T//2; abrupt switches before query t=T//2.",
                "initial_conflicts": "A configured fraction of project-attribute pairs begin with two distinct values.",
                "updates_and_repetitions": "Each round writes a new value with update_probability; otherwise repeats the current value.",
                "latest_evidence": "The latest statement is evidence even when it repeats the current value.",
                "external_data_or_llm": False,
            },
            "statistical_unit": "independent dialogue; queries within a dialogue are dependent",
            "episodes": [
                {"seed": episode["seed"], "scenario": episode["scenario"], "questions": len(episode["probes"])}
                for episode in episodes
            ],
        },
    )
    save_json(
        checks_path,
        {
            "status": "passed",
            "episode_count": len(episodes),
            "question_count": sum(record["question_count"] for record in audit_records),
            "all_questions_independently_reconstructed": True,
            "episodes": audit_records,
        },
    )
    save_csv(profile_path, profile)
    return {
        "run_dir": str(run_dir),
        "episodes": len(episodes),
        "questions": int(sum(record["question_count"] for record in audit_records)),
        "detail_profile": detail,
        "profile": profile,
        "files": [str(path) for path in targets],
    }


In [ ]:
import gzip as _m1_test_gzip
import json as _m1_test_json
import tempfile as _m1_test_tempfile
from pathlib import Path as _m1_test_Path

M1_TEST_CONFIG = {
    "T": 8,
    "n_projects": 3,
    "p_recent_stable": 0.9,
    "p_recent_abrupt_before": 0.9,
    "p_recent_abrupt_after": 0.1,
    "update_probability": 0.25,
    "initial_conflict_fraction": 0.25,
    "alias_probability": 0.5,
}

M1_FIXTURE = {
    "seed": 7,
    "scenario": "stable",
    "history": [
        {"id": 0, "text": "Focus is project P1. We are reviewing its latest requirements."},
        {"id": 1, "text": "For project P1: the dataset is V1. " + M1_SYSTEM_CONTEXT},
        {"id": 2, "text": "For project P1: the data source is V2. " + M1_SYSTEM_CONTEXT},
        {"id": 3, "text": "The next meeting will review progress and remaining questions."},
        {"id": 4, "text": "Focus is project P2. We are reviewing its latest requirements."},
        {"id": 5, "text": "For project P2: the language is V3. " + M1_SYSTEM_CONTEXT},
        {"id": 6, "text": "For project P1: the dataset is V2. " + M1_SYSTEM_CONTEXT},
    ],
    "probes": [
        {
            "t": 0,
            "question": "What is its current data source?",
            "end": 4,
            "gold": "V2",
            "evidence": 2,
            "old_values": ["V1"],
            "qtype": "recent",
            "state": "A",
        },
        {
            "t": 1,
            "question": "What is P1 current dataset?",
            "end": 6,
            "gold": "V2",
            "evidence": 2,
            "old_values": ["V1"],
            "qtype": "historical",
            "state": "A",
        },
        {
            "t": 2,
            "question": "What is P1 current dataset?",
            "end": 7,
            "gold": "V2",
            "evidence": 6,
            "old_values": ["V1"],
            "qtype": "historical",
            "state": "A",
        },
    ],
}

_m1_test_stable_a = generate_episode(31415, "stable", M1_TEST_CONFIG)
_m1_test_stable_b = generate_episode(31415, "stable", M1_TEST_CONFIG)
assert _m1_test_stable_a == _m1_test_stable_b, "same seed/config must reproduce the episode"
assert _m1_test_stable_a != generate_episode(31416, "stable", M1_TEST_CONFIG)
assert [m["id"] for m in _m1_test_stable_a["history"]] == list(
    range(len(_m1_test_stable_a["history"]))
)
assert [p["t"] for p in _m1_test_stable_a["probes"]] == list(range(8))
assert audit_truth(_m1_test_stable_a)["passed"]

_m1_test_abrupt = generate_episode(31415, "abrupt", M1_TEST_CONFIG)
assert [p["state"] for p in _m1_test_abrupt["probes"]] == ["A"] * 4 + ["B"] * 4
assert audit_truth(_m1_test_abrupt)["passed"]

_m1_test_probe = _m1_test_stable_a["probes"][0]
_m1_test_view = public_view(_m1_test_stable_a, _m1_test_probe)
assert set(_m1_test_view) == {"question", "history"}
assert len(_m1_test_view["history"]) == _m1_test_probe["end"]
assert all(set(m) == {"id", "text"} for m in _m1_test_view["history"])
_m1_test_future_variant = _m1_test_json.loads(_m1_test_json.dumps(_m1_test_stable_a))
_m1_test_future_variant["history"][_m1_test_probe["end"]]["text"] = "Future-only mutation."
assert public_view(_m1_test_future_variant, _m1_test_future_variant["probes"][0]) == _m1_test_view

_m1_test_fixture_audit = audit_truth(M1_FIXTURE)
assert _m1_test_fixture_audit["passed"]
assert _m1_test_fixture_audit["questions"][0]["derived_gold"] == "V2"
assert _m1_test_fixture_audit["questions"][0]["derived_evidence"] == 2
assert _m1_test_fixture_audit["questions"][2]["derived_evidence"] == 6, (
    "a repeated value must still move evidence to its latest statement"
)

_m1_test_corrupt = _m1_test_json.loads(_m1_test_json.dumps(M1_FIXTURE))
_m1_test_corrupt["probes"][0]["gold"] = "V999"
_m1_test_corrupt["probes"][0]["evidence"] = 1
_m1_test_corrupt["probes"][0]["old_values"] = []
_m1_test_corrupt_audit = audit_truth(_m1_test_corrupt)
assert not _m1_test_corrupt_audit["passed"]
assert len(_m1_test_corrupt_audit["questions"][0]["errors"]) >= 3
_m1_test_future_evidence = _m1_test_json.loads(_m1_test_json.dumps(M1_FIXTURE))
_m1_test_future_evidence["probes"][0]["evidence"] = 6
_m1_future_audit = audit_truth(_m1_test_future_evidence)
assert any("strictly inside" in error for error in _m1_future_audit["questions"][0]["errors"])

try:
    generate_episode(1, "unsupported", M1_TEST_CONFIG)
except ValueError:
    pass
else:
    raise AssertionError("unsupported scenarios must fail clearly")

with _m1_test_tempfile.TemporaryDirectory(prefix="m1_fixture_", dir=OUTPUTS_DIR) as _m1_temp:
    _m1_test_run = _m1_test_Path(_m1_temp)
    for _m1_area in ("data", "tables", "audit"):
        (_m1_test_run / _m1_area).mkdir()
    _m1_export = export_development_data([M1_FIXTURE], _m1_test_run)
    assert _m1_export["episodes"] == 1 and _m1_export["questions"] == 3
    _m1_data_path = _m1_test_run / "data" / "development_dialogues.jsonl.gz"
    with _m1_test_gzip.open(_m1_data_path, "rt", encoding="utf-8") as _m1_stream:
        _m1_roundtrip = _m1_test_json.loads(_m1_stream.readline())
    assert _m1_roundtrip == M1_FIXTURE
    try:
        export_development_data([M1_FIXTURE], _m1_test_run)
    except FileExistsError:
        pass
    else:
        raise AssertionError("M1 exporter must refuse to overwrite existing artifacts")
    assert len(list(_m1_test_run.rglob("*"))) >= 7

M1_TEST_RESULTS = {
    "status": "passed",
    "same_seed_reproducibility": True,
    "different_seed_changes_episode": True,
    "continuous_ids_and_time": True,
    "abrupt_switch_at_T_half": True,
    "public_view_hides_evaluator_fields_and_future": True,
    "independent_parser_handles_its_alias_explicit_project_and_versions": True,
    "repeated_value_moves_latest_evidence": True,
    "audit_detects_corrupt_gold_evidence_and_old_values": True,
    "audit_rejects_future_evidence": True,
    "export_roundtrip_and_no_overwrite": True,
    "fixture_export_is_temporary_and_excluded_from_development_data": True,
}
print("M1 checks passed:", _m1_test_json.dumps(M1_TEST_RESULTS, sort_keys=True))

# Set this True only after choosing a new development RUN_ID in the shared paths cell.
M1_EXPORT_DEVELOPMENT = False
if M1_EXPORT_DEVELOPMENT:
    if MODE != "development" or RUN_ID == "local_setup":
        raise RuntimeError("Choose MODE='development' and a new, explicit M1 RUN_ID first")
    _m1_dev_episodes = [
        generate_episode(int(PROTOCOL_DRAFT["dev_seed"]) + 100 * scenario_index + index,
                         scenario, PROTOCOL_DRAFT)
        for scenario_index, scenario in enumerate(SCENARIOS)
        for index in range(int(PROTOCOL_DRAFT["n_dev_per_scenario"]))
    ]
    M1_DEVELOPMENT_EXPORT = export_development_data(_m1_dev_episodes, RUN_DIR)
    print("M1 development export:", M1_DEVELOPMENT_EXPORT["run_dir"])


### M1研究记录（自动化实现后待成员复核）

本轮按共同草案实现6个项目、4种属性和每对话300题；更新、复述、别名和初始多版本规则见 `deliverables/M1_notes.md`。独立解析器从每题公开历史前缀重建值与证据，不使用保存的 `gold` 作为推导来源。

Development run `outputs/development/m1_development_20261109` 包含6条对话、1,800题；1,800题自动真值核查通过，gzip数据读回一致。实测近期题比例：stable/A 0.890、abrupt/A 0.893、abrupt/B 0.102；后半段旧值比例0.898，平均证据距离72.362条消息。比例是随机样本观察，不是策略效果结论。

自动测试覆盖seed复现、连续ID、信息前缀、切换边界、更新/复述/别名、真值篡改检测和无覆盖导出。测试通过不等于人工审查或团队验收。

M2四臂材料和M1/M2十题双签尚未完成；M2/M4接收待确认。姓名留待成员本人填写。代码和记录由Codex按项目负责人的授权协助完成，成员需亲自复跑、理解并确认AI辅助说明后再署名。


<a id="member-m2"></a>
## M2 · 策略、回答器与成本
**四臂选择、规则回答与评价代码由你编写。任务区间m2-*。**
输入：M1公开视图与episode；上游未完成时自建短历史。输出给M3/M4，审计材料交M1。

1. **定义公平比较。** 研究共同输入模板、计数规则、UNKNOWN与最新事实优先规则；四臂必须共享回答器与成本口径。
2. **逐臂实现。** 从连续窗口到TF–IDF，再实现仅用公开历史的指代消解和同义词处理；记录平局、超预算消息、focus锚点处理。
3. **写自己的测试。** 检查预算边界、空历史、未来隔离、时序、旧值；不可读取gold/隐藏题型。
4. **评价与研究。** 为每题每臂导出行，记录完整输入成本和候选构造耗时，研究accuracy/coverage/outdated的关系；结果由实验决定。
5. **交接。** 导出10题的公开历史、各臂上下文/回答/成本和评估真值；M1独立追踪，你交叉复核。先给M3单对话表，再给M4全部开发表。

**验收方向：** 预算只管上下文；C含共同指令、上下文、问题；id唯一且时序；完整/空历史行为有依据。至少三个自己发现或构造的边界案例及测试局限。
**输出：** tables/development_outcomes.csv、strategy_by_query.csv；audit/audit_examples.json、audit_review.csv、strategy_checks.json。人工审计不能自动填pass。


### M2 第一个里程碑 · 从短历史走通四臂评价

按顺序运行第0节、共享协议和M1，再运行本节。手工 fixture 也使用 M1 的公开视图接口。
本节只使用明确标为 **fixture** 的9条消息、4个问题，不把它们写成正式开发集或实验结论。

| 步骤 | 阅读重点 | 可见结果 |
|---|---|---|
| A | 统一计数与输入模板 | 一段文字如何变成成本单位 |
| B | 事实、同义词、focus与回答器 | 全历史的手工答案对照 |
| C | 预算打包与四种策略 | 各臂选中消息、预算、TF–IDF排序 |
| D | 独立评价与成本分解 | 16行四臂表、完整输入示例 |
| E | 边界检查 | 预算、时序、空输入、未来隔离与指代反例 |

参考来源：[v1完整参考本](reference/ARIN7101_Context_Bandit_Release_v1_Complete_Reference.ipynb) 的
`data-generator` 中解析规则、`strategy-evaluation` 中窗口/检索/打包思路及 `shared-config` 中正则计数。
这里重新组织为可分步运行的模块，显式处理空词表、校验公开视图，并通过独立手写预期值检查样例。


#### A · 统一计数与输入模板

使用 `\w+|[^\w\s]`：每个连续词串或独立标点计1单位，空白不计。
上下文预算只计算消息正文；总成本还包括共同指令、`Context:` / `Question:` 标签和问题。
四臂共用同一模板。这是长度代理，不是真实token、API费用或延迟。


In [ ]:
import re
from copy import deepcopy
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

M2_INSTRUCTION = (
    "Answer using only the context. Later facts override earlier facts. "
    "Resolve its using the latest focus. Return the value, or UNKNOWN if unsupported."
)


def count_input_units(text):
    """正则长度代理；例如 P1: V11. 被分为 P1 / : / V11 / .。"""
    return len(re.findall(r"\w+|[^\w\s]", text))


def m2_context_text(context):
    return "\n".join(message["text"] for message in context)


def m2_build_prompt(question, context):
    return M2_INSTRUCTION + "\nContext:\n" + m2_context_text(context) + "\nQuestion: " + question


sample_text = "P1: V11."
display(pd.DataFrame([{
    "text": sample_text,
    "regex_units": re.findall(r"\w+|[^\w\s]", sample_text),
    "count": count_input_units(sample_text),
}]))


#### B · 从公开文本识别对象、属性和最新事实

先把 `data source / coding language / evaluation measure / due date` 归一化为四种属性。
显式项目名优先；只有问题包含 `its` 时才使用**所给上下文**中的最新focus。
回答器依消息顺序更新事实，最后一个版本覆盖旧版本；缺对象、属性或事实都返回 `UNKNOWN`。
回答器不接收gold、evidence或完整episode。


In [ ]:
M2_ALIASES = {
    "dataset": "data source", "language": "coding language",
    "metric": "evaluation measure", "deadline": "due date",
}
M2_PROJECT = re.compile(r"\bP\d+\b", re.I)
M2_FOCUS = re.compile(r"\bFocus is project (P\d+)\b", re.I)
M2_FACT = re.compile(
    r"\bproject (P\d+): the (dataset|language|metric|deadline) is (V\d+)\b", re.I
)


def m2_normalize(text):
    for canonical, alias in M2_ALIASES.items():
        text = re.sub(r"\b" + re.escape(alias) + r"\b", canonical, text, flags=re.I)
    return text


def m2_parse_fact(text):
    match = M2_FACT.search(m2_normalize(text))
    if match is None:
        return None
    project, attribute, value = match.groups()
    return project.upper(), attribute.lower(), value.upper()


def m2_resolve_target(question, context):
    explicit = M2_PROJECT.search(question)
    project = explicit.group().upper() if explicit else None
    if project is None and re.search(r"\bits\b", question, re.I):
        for message in context:
            focus = M2_FOCUS.search(message["text"])
            if focus:
                project = focus.group(1).upper()
    normalized = m2_normalize(question)
    attribute = next((name for name in M2_ALIASES
                      if re.search(r"\b" + name + r"\b", normalized, re.I)), None)
    return project, attribute


def rule_reader(question, context):
    """只使用调用者给定的、按时间排列的上下文。"""
    target = m2_resolve_target(question, context)
    answer = "UNKNOWN"
    for message in context:
        fact = m2_parse_fact(message["text"])
        if fact and fact[:2] == target:
            answer = fact[2]
    return answer


In [ ]:
# 手工fixture：id 0/3/8 是P1的旧值、更新、同值复述；id 2/5 是focus切换。
M2_FIXTURE = {
    "seed": 42, "scenario": "fixture",
    "history": [
        {"id": 0, "text": "For project P1: the dataset is V10."},
        {"id": 1, "text": "For project P2: the language is V20."},
        {"id": 2, "text": "Focus is project P1. We are reviewing its latest requirements."},
        {"id": 3, "text": "For project P1: the data source is V11."},
        {"id": 4, "text": "The next meeting will review progress and remaining questions."},
        {"id": 5, "text": "Focus is project P2. We are reviewing its latest requirements."},
        {"id": 6, "text": "For project P2: the coding language is V21."},
        {"id": 7, "text": "The next meeting will review progress and remaining questions."},
        {"id": 8, "text": "For project P1: the data source is V11."},
    ],
    "probes": [
        {"t": 0, "question": "What is P1 current dataset?", "end": 3,
         "gold": "V10", "evidence": 0, "old_values": [], "qtype": "historical", "state": "A"},
        {"t": 1, "question": "What is its current data source?", "end": 5,
         "gold": "V11", "evidence": 3, "old_values": ["V10"], "qtype": "recent", "state": "A"},
        {"t": 2, "question": "What is P1 current dataset?", "end": 8,
         "gold": "V11", "evidence": 3, "old_values": ["V10"], "qtype": "historical", "state": "A"},
        {"t": 3, "question": "What is its current coding language?", "end": 9,
         "gold": "V21", "evidence": 6, "old_values": ["V20"], "qtype": "recent", "state": "A"},
    ],
}
M2_CFG = deepcopy(PROTOCOL_DRAFT)
display(pd.DataFrame(M2_FIXTURE["history"]))
display(pd.DataFrame(M2_FIXTURE["probes"])[
    ["t", "end", "question", "gold", "evidence", "old_values"]
])
print("end是Python切片的右开边界：t=0只能看id 0、1、2，不能看id 3的更新。")


In [ ]:
# 此处的切片只是fixture演示，不实现或覆盖M1的public_view。
reader_examples = []
for probe in M2_FIXTURE["probes"]:
    prefix = M2_FIXTURE["history"][:probe["end"]]
    project, attribute = m2_resolve_target(probe["question"], prefix)
    reader_examples.append({
        "t": probe["t"], "project": project, "attribute": attribute,
        "visible_ids": [m["id"] for m in prefix],
        "full_history_answer": rule_reader(probe["question"], prefix),
        "manual_gold": probe["gold"],
        "empty_context_answer": rule_reader(probe["question"], []),
    })
display(pd.DataFrame(reader_examples))
assert [r["full_history_answer"] for r in reader_examples] == ["V10", "V11", "V11", "V21"]


#### C · 先打包，再比较四臂

- recent两臂从最新消息向前选，遇到第一条放不下的消息就停止，以保留连续后缀；short最多检查3条。
- 检索臂按“相似度降序、平局时较新ID优先”保留最多8个候选，放不下的候选跳过，最后按ID升序返回。
- resolved臂只从公开历史识别对象、归一化同义词；`its`题优先打包最新focus，它可以在8个检索候选之外。
- 全部消息按整条计费，不截断正文；返回的ID必须唯一且升序。空历史或TF–IDF空词表返回空上下文。

最新focus若自身放不下，resolved臂返回空上下文，避免借用旧focus；不会把全历史消解出的对象偷偷传给回答器。


In [ ]:
def m2_validate_history(history):
    ids = [message["id"] for message in history]
    if any(type(message_id) is not int or message_id < 0 for message_id in ids):
        raise ValueError("消息id必须是非负整数。")
    if ids != sorted(set(ids)):
        raise ValueError("输入消息id必须唯一且按时间升序。")
    if any(not isinstance(message["text"], str) for message in history):
        raise ValueError("消息text必须是字符串。")


def m2_pack_messages(history, positions, budget, stop_on_fail=False):
    """positions是候选在history中的位置；返回消息副本，不修改输入。"""
    chosen, seen, used = [], set(), 0
    for position in positions:
        message = history[position]
        if message["id"] in seen:
            continue
        seen.add(message["id"])
        units = count_input_units(message["text"])
        if used + units <= budget:
            chosen.append(dict(message))
            used += units
        elif stop_on_fail:
            break
    return sorted(chosen, key=lambda message: message["id"])


In [ ]:
def m2_rank_tfidf(view, resolved=False):
    """返回检索诊断表；只在当前公开前缀拟合词表。"""
    history, question = view["history"], view["question"]
    columns = ["position", "id", "score", "query", "document"]
    if not history:
        return pd.DataFrame(columns=columns)
    documents = [message["text"] for message in history]
    if resolved:
        project, attribute = m2_resolve_target(question, history)
        question = " ".join(part for part in (project, attribute) if part)
        documents = [m2_normalize(text) for text in documents]
    # 使用和TfidfVectorizer默认设置相同的分词规则，显式处理全标点/空文本。
    vectorizer = TfidfVectorizer()
    analyzer = vectorizer.build_analyzer()
    if not any(analyzer(document) for document in documents):
        return pd.DataFrame(columns=columns)
    matrix = vectorizer.fit_transform(documents)
    scores = (matrix @ vectorizer.transform([question]).T).toarray().ravel()
    ranking = pd.DataFrame([
        {"position": i, "id": message["id"], "score": float(scores[i]),
         "query": question, "document": documents[i]}
        for i, message in enumerate(history)
    ])
    return ranking.sort_values(["score", "id"], ascending=[False, False]).reset_index(drop=True)


In [ ]:
def select_context(view, arm, cfg):
    """接口仅接受question/history；预算只约束消息正文。"""
    if set(view) != {"question", "history"}:
        raise ValueError("公开视图只能包含question/history。")
    if arm not in ARMS:
        raise ValueError(f"未知策略：{arm}")
    history = view["history"]
    m2_validate_history(history)
    budget = cfg["budgets"][arm]
    if type(budget) is not int or budget < 0:
        raise ValueError("上下文预算必须是非负整数。")
    if not history or budget == 0:
        return []
    if arm in ("recent_short", "recent_long"):
        positions = list(range(len(history) - 1, -1, -1))
        if arm == "recent_short":
            positions = positions[:3]
        return m2_pack_messages(history, positions, budget, stop_on_fail=True)

    resolved = arm == "resolved_tfidf"
    ranking = m2_rank_tfidf(view, resolved=resolved)
    positions = ranking["position"].head(8).tolist()
    if resolved and not M2_PROJECT.search(view["question"]) and re.search(r"\bits\b", view["question"], re.I):
        anchors = [i for i, message in enumerate(history) if M2_FOCUS.search(message["text"])]
        if anchors:
            anchor = anchors[-1]
            if count_input_units(history[anchor]["text"]) > budget:
                return []
            positions = [anchor] + [i for i in positions if i != anchor]
    return m2_pack_messages(history, positions, budget)


In [ ]:
# 观察t=3：short虽然拿到了language事实(id 6)，却丢失了focus(id 5)。
M2_DEMO_PROBE = M2_FIXTURE["probes"][3]
M2_DEMO_VIEW = {
    "question": M2_DEMO_PROBE["question"],
    "history": deepcopy(M2_FIXTURE["history"][:M2_DEMO_PROBE["end"]]),
}
for arm in ARMS:
    context = select_context(M2_DEMO_VIEW, arm, M2_CFG)
    units = count_input_units(m2_context_text(context))
    print(f"{arm}: context={units}/{M2_CFG['budgets'][arm]}, "
          f"answer={rule_reader(M2_DEMO_VIEW['question'], context)}")
    display(pd.DataFrame(context, columns=["id", "text"]))

print("原问题检索排序（前8名）：")
display(m2_rank_tfidf(M2_DEMO_VIEW).head(8).round({"score": 4}))
print("公开历史消解后的检索排序（前8名；另优先打包focus）：")
display(m2_rank_tfidf(M2_DEMO_VIEW, resolved=True).head(8).round({"score": 4}))


#### D · 评价器读取真值，策略与回答器只读取公开输入

`evaluate_episode(episode, cfg)`为每题每臂返回一行，字段与输出契约一致。
当前M1的`public_view`仍是占位函数，所以本里程碑由评价器显式构造同契约的公开前缀；待联调时与M1实现交叉核对。
这里定义 `end` 为右开切片边界，不从隐藏标签推断策略。

`coverage`检查evidence的消息ID是否入选；`outdated`检查回答是否属于old_values；
`y`对照gold；`utility = y - cfg["lambda"] * cost / cfg["cost_reference"]`。
`construction_ms`只测上下文构造，受本机运行状态影响，不要求重复运行完全一致。


In [ ]:
M2_OUTCOME_COLUMNS = [
    "seed", "scenario", "t", "arm", "question", "y", "cost", "utility",
    "coverage", "outdated", "construction_ms", "qtype", "state",
]


def evaluate_episode(episode, cfg):
    """真值只在策略选择和回答之后用于评分；不自动导出或宣称审计通过。"""
    m2_validate_history(episode["history"])
    rows = []
    for probe in episode["probes"]:
        view = public_view(episode, probe)
        for arm in ARMS:
            start = perf_counter()
            context = select_context(view, arm, cfg)
            elapsed_ms = (perf_counter() - start) * 1000
            ids = [message["id"] for message in context]
            if ids != sorted(set(ids)) or not set(ids) <= {m["id"] for m in view["history"]}:
                raise ValueError("策略返回了重复、乱序或非公开消息。")
            if count_input_units(m2_context_text(context)) > cfg["budgets"][arm]:
                raise ValueError("上下文超过预算。")
            answer = rule_reader(view["question"], context)
            cost = count_input_units(m2_build_prompt(view["question"], context))
            correct = int(answer == probe["gold"])
            rows.append({
                "seed": episode["seed"], "scenario": episode["scenario"], "t": probe["t"],
                "arm": arm, "question": probe["question"], "y": correct, "cost": cost,
                "utility": correct - cfg["lambda"] * cost / cfg["cost_reference"],
                "coverage": int(probe["evidence"] in ids),
                "outdated": int(answer in probe["old_values"]),
                "construction_ms": elapsed_ms, "qtype": probe["qtype"], "state": probe["state"],
            })
    result = pd.DataFrame(rows, columns=M2_OUTCOME_COLUMNS)
    if result.duplicated(["seed", "t", "arm"]).any():
        raise ValueError("四臂表seed/t/arm不唯一。")
    return result


In [ ]:
def m2_inspect_probe(episode, probe, cfg):
    """演示用额外诊断列，和交给M3的固定schema分开。"""
    view = public_view(episode, probe)
    rows = []
    for arm in ARMS:
        context = select_context(view, arm, cfg)
        parts = {
            "instruction_units": count_input_units(M2_INSTRUCTION),
            "format_units": count_input_units("Context:\nQuestion: "),
            "context_units": count_input_units(m2_context_text(context)),
            "question_units": count_input_units(view["question"]),
        }
        cost = count_input_units(m2_build_prompt(view["question"], context))
        assert sum(parts.values()) == cost
        rows.append({
            "t": probe["t"], "arm": arm,
            "selected_ids": [message["id"] for message in context],
            "answer": rule_reader(view["question"], context), "gold": probe["gold"],
            "evidence": probe["evidence"], "budget": cfg["budgets"][arm],
            **parts, "cost": cost,
        })
    return pd.DataFrame(rows)


M2_FIXTURE_DETAILS = pd.concat([
    m2_inspect_probe(M2_FIXTURE, probe, M2_CFG) for probe in M2_FIXTURE["probes"]
], ignore_index=True)
display(M2_FIXTURE_DETAILS)

demo_context = select_context(M2_DEMO_VIEW, "recent_short", M2_CFG)
print("t=3 / recent_short 的完整输入，可对照上表逐项数成本：")
print(m2_build_prompt(M2_DEMO_VIEW["question"], demo_context))


In [ ]:
M2_FIXTURE_OUTCOMES = evaluate_episode(M2_FIXTURE, M2_CFG)
display(M2_FIXTURE_OUTCOMES.round({"utility": 6, "construction_ms": 3}))

# 将固定接口表与手工核对信息并排展示；不把诊断列混入M3的接口表。
comparison = M2_FIXTURE_OUTCOMES.merge(
    M2_FIXTURE_DETAILS[["t", "arm", "selected_ids", "answer", "gold", "context_units"]],
    on=["t", "arm"], validate="one_to_one",
)
display(comparison[[
    "t", "arm", "selected_ids", "answer", "gold", "y", "coverage", "outdated", "cost", "utility"
]].round({"utility": 6}))
print("fixture共4题 × 4臂 =", len(M2_FIXTURE_OUTCOMES), "行；未生成真实开发集或人工审计结论。")


#### E · 用可定位的边界案例验证性质

下面的检查不估计总体性能；它们针对具体实现风险。
预算测试使用刻意缩小的配置副本，不改公共设计。未来隔离测试保持当前公开前缀不变，仅改未来消息；
元数据隔离测试单独改变gold/隐藏标签，确认评价器传给选择器的公开输入不变。


In [ ]:
M2_CHECKS = []

# 1. 手写预期：short在t=2找不到P1事实，在t=3找不到its的focus。
short = M2_FIXTURE_OUTCOMES.query("arm == 'recent_short'").sort_values("t")
assert short["y"].tolist() == [1, 1, 0, 0]
assert short["coverage"].tolist() == [1, 1, 0, 1]
assert M2_FIXTURE_DETAILS.query("arm == 'recent_short'")["answer"].tolist() == [
    "V10", "V11", "UNKNOWN", "UNKNOWN"
]
assert len(M2_FIXTURE_OUTCOMES) == 16
assert M2_FIXTURE_OUTCOMES.columns.tolist() == M2_OUTCOME_COLUMNS
assert not M2_FIXTURE_OUTCOMES.duplicated(["seed", "t", "arm"]).any()
M2_CHECKS.append({"case": "手写答案/coverage/16行唯一键", "result": "pass"})

# 2. 恰好预算3、超出预算2；长窗遇到超长末条就停，检索打包允许跳过。
tiny = [{"id": 0, "text": "a b"}, {"id": 1, "text": "c"}]
tiny_cfg = deepcopy(M2_CFG)
tiny_cfg["budgets"]["recent_long"] = 3
assert [m["id"] for m in select_context({"question": "?", "history": tiny}, "recent_long", tiny_cfg)] == [0, 1]
tiny_cfg["budgets"]["recent_long"] = 2
assert [m["id"] for m in select_context({"question": "?", "history": tiny}, "recent_long", tiny_cfg)] == [1]
oversize = [{"id": 0, "text": "a"}, {"id": 1, "text": "b c d"}]
assert m2_pack_messages(oversize, [1, 0], 2, stop_on_fail=True) == []
assert [m["id"] for m in m2_pack_messages(oversize, [1, 0], 2)] == [0]
assert [m["id"] for m in m2_pack_messages(tiny, [1, 1, 0], 3)] == [0, 1]
M2_CHECKS.append({"case": "预算边界/整条打包/重复候选", "result": "pass"})

# 3. 空历史、全标点词表、无focus；检查所有臂的输出约束。
for arm in ARMS:
    assert select_context({"question": "What is its dataset?", "history": []}, arm, M2_CFG) == []
    for probe in M2_FIXTURE["probes"]:
        view = public_view(M2_FIXTURE, probe)
        context = select_context(view, arm, M2_CFG)
        ids = [m["id"] for m in context]
        assert ids == sorted(set(ids)) and all(i < probe["end"] for i in ids)
        assert count_input_units(m2_context_text(context)) <= M2_CFG["budgets"][arm]
punctuation = {"question": "?", "history": [{"id": 0, "text": "..."}]}
assert select_context(punctuation, "tfidf", M2_CFG) == []
assert select_context(punctuation, "resolved_tfidf", M2_CFG) == []
assert rule_reader("What is its dataset?", [M2_FIXTURE["history"][3]]) == "UNKNOWN"
M2_CHECKS.append({"case": "空历史/空词表/缺focus/预算和时序", "result": "pass"})


In [ ]:
# 4. 新旧版本和复述由手写常量验证，而不是拿回答器证明自己的输出。
question = "What is P1 current dataset?"
history = M2_FIXTURE["history"]
assert rule_reader(question, [history[0]]) == "V10"
assert rule_reader(question, [history[0], history[3]]) == "V11"
assert rule_reader(question, [history[3], history[8]]) == "V11"
assert rule_reader(question, []) == "UNKNOWN"
assert rule_reader("What is P1 current due date?", history) == "UNKNOWN"
# 合法的两消息历史：新事实过长、超出所有臂预算；检索跳过它而返回旧值。
old_episode = {
    "seed": 43, "scenario": "fixture",
    "history": [dict(history[0]), {"id": 1, "text": history[3]["text"] + " extra" * 600}],
    "probes": [dict(M2_FIXTURE["probes"][2], t=0, end=2, evidence=1)],
}
old_rows = evaluate_episode(old_episode, M2_CFG)
assert old_rows["y"].tolist() == [0, 0, 0, 0]
assert old_rows["outdated"].tolist() == [0, 0, 1, 1]
M2_CHECKS.append({"case": "旧值/更新/同值复述/UNKNOWN/outdated评分", "result": "pass"})

# 5. 同一公开前缀，未来事实不同，当前结果除耗时外完全不变。
before = deepcopy(M2_FIXTURE)
before["probes"] = [before["probes"][1]]
changed_future = deepcopy(before)
changed_future["history"][6]["text"] = "For project P1: the dataset is V999."
changed_future["history"].append({"id": 9, "text": "Focus is project P999."})
pd.testing.assert_frame_equal(
    evaluate_episode(before, M2_CFG).drop(columns="construction_ms"),
    evaluate_episode(changed_future, M2_CFG).drop(columns="construction_ms"),
)
changed_labels = deepcopy(before)
changed_labels["probes"][0].update(gold="V999", qtype="hidden_changed", state="hidden_changed")
assert public_view(before, before["probes"][0]) == public_view(
    changed_labels, changed_labels["probes"][0]
)
assert before == dict(deepcopy(M2_FIXTURE), probes=[deepcopy(M2_FIXTURE["probes"][1])])
M2_CHECKS.append({"case": "未来隔离/隐藏标签隔离/输入未被修改", "result": "pass"})

# 6. 最新focus自身超预算时，resolved不带着旧focus去回答。
focus_view = {"question": "What is its dataset?", "history": [
    {"id": 0, "text": "Focus is project P1."},
    {"id": 1, "text": "For project P1: the dataset is V10."},
    {"id": 2, "text": "Focus is project P2. " + "extra " * 30},
]}
small_cfg = deepcopy(M2_CFG)
small_cfg["budgets"]["resolved_tfidf"] = 20
assert select_context(focus_view, "resolved_tfidf", small_cfg) == []
assert count_input_units("P1: V11.") == 4
np.testing.assert_allclose(
    M2_FIXTURE_OUTCOMES["utility"],
    M2_FIXTURE_OUTCOMES["y"] - 0.5 * M2_FIXTURE_OUTCOMES["cost"] / 512,
)
M2_CHECKS.append({"case": "focus锚点预算/正则计数/效用公式", "result": "pass"})

# 7. 十条完全同分的消息应优先取最新8条，最终恢复时间顺序。
tie_view = {"question": "dataset", "history": [
    {"id": i, "text": "dataset"} for i in range(10)
]}
assert [m["id"] for m in select_context(tie_view, "tfidf", M2_CFG)] == list(range(2, 10))
for bad_view in [
    dict(M2_DEMO_VIEW, gold="V21"),
    {"question": "?", "history": [{"id": 0, "text": "a"}, {"id": 0, "text": "b"}]},
]:
    try:
        select_context(bad_view, "tfidf", M2_CFG)
    except ValueError:
        pass
    else:
        raise AssertionError("隐藏字段或重复ID没有被拒绝。")
M2_CHECKS.append({"case": "检索同分/8候选上限/非法公开输入", "result": "pass"})
display(pd.DataFrame(M2_CHECKS))


In [ ]:
M2_DEVELOPMENT_AUDIT_KEYS = (
    (130000, 0), (130000, 2), (130000, 12), (130001, 1),
    (130100, 99), (130100, 149), (130100, 150), (130100, 151),
    (130101, 149), (130101, 299),
)


def m2_validate_development_outcomes(episodes, outcomes, cfg):
    """核对完整开发表与 M1 输入的一一对应关系。"""
    if not episodes or list(outcomes.columns) != M2_OUTCOME_COLUMNS:
        raise ValueError("需要非空的 M1 开发对话和符合契约的四臂表。")
    seeds = [episode["seed"] for episode in episodes]
    if len(seeds) != len(set(seeds)) or any(ep["scenario"] not in SCENARIOS for ep in episodes):
        raise ValueError("开发对话须属于 stable/abrupt，且 seed 在整个 run 中唯一。")
    identity = ["seed", "t", "arm"]
    labels = ["scenario", "question", "qtype", "state"]
    expected = pd.DataFrame(
        (
            (ep["seed"], probe["t"], arm, ep["scenario"],
             probe["question"], probe["qtype"], probe["state"])
            for ep in episodes for probe in ep["probes"] for arm in ARMS
        ),
        columns=identity + labels,
    )
    actual = outcomes[identity + labels]
    if (expected.empty or expected.duplicated(identity).any()
            or not actual.sort_values(identity, ignore_index=True).equals(
                expected.sort_values(identity, ignore_index=True)
            )):
        raise ValueError("四臂表的唯一键、每题臂集合或题目元数据与 M1 不一致。")
    if not outcomes[["y", "coverage", "outdated"]].isin([0, 1]).all().all():
        raise ValueError("正确性、覆盖和过期值指标须为 0/1。")
    cost_and_time = outcomes[["cost", "construction_ms"]]
    if not np.isfinite(cost_and_time).all().all() or (cost_and_time < 0).any().any():
        raise ValueError("成本和构造耗时须为有限非负数。")
    if not outcomes["cost"].mod(1).eq(0).all():
        raise ValueError("完整输入成本须为整数计数。")
    if not np.allclose(outcomes["utility"], outcomes["y"] - cfg["lambda"] * outcomes["cost"] / cfg["cost_reference"]):
        raise ValueError("效用与当前成本配置不一致。")


def m2_build_audit_examples(episodes, outcomes, cfg, audit_keys):
    """从用于评价的同一份公开输入重建审计证据。"""
    if len(audit_keys) != 10 or len(set(audit_keys)) != 10:
        raise ValueError("须明确给出十个不同的审计题键。")
    probes = {(ep["seed"], probe["t"]): (ep, probe) for ep in episodes for probe in ep["probes"]}
    if not set(audit_keys) <= probes.keys():
        raise ValueError("审计题键必须存在于本次开发对话。")
    indexed = outcomes.set_index(["seed", "t", "arm"])
    examples = []
    for seed, t in audit_keys:
        episode, probe = probes[seed, t]
        view = public_view(episode, probe)
        history = {message["id"]: message for message in view["history"]}
        evidence = probe["evidence"]
        latest_fact = m2_parse_fact(history[evidence]["text"])
        if latest_fact is None:
            raise ValueError(f"无法解析审计题的最新证据 seed={seed}, t={t}。")
        previous = next((
            fact for message in reversed(view["history"])
            if message["id"] < evidence and (fact := m2_parse_fact(message["text"]))
            and fact[:2] == latest_fact[:2]
        ), None)
        fact_event = "initial" if previous is None else "repeat" if previous[2] == latest_fact[2] else "update"
        arms = []
        for details in m2_inspect_probe(episode, probe, cfg).to_dict("records"):
            arm = details.pop("arm")
            selected_ids = details.pop("selected_ids")
            answer = details["answer"]
            metrics = {
                "y": int(answer == probe["gold"]),
                "cost": details["cost"],
                "coverage": int(evidence in selected_ids),
                "outdated": int(answer in probe["old_values"]),
            }
            metrics["utility"] = metrics["y"] - cfg["lambda"] * metrics["cost"] / cfg["cost_reference"]
            stored = indexed.loc[(seed, t, arm)]
            if any(not np.isclose(value, stored[name]) for name, value in metrics.items()):
                raise ValueError(f"重建证据与四臂表不一致 seed={seed}, t={t}, arm={arm}。")
            parts = {key: value for key, value in details.items() if key.endswith("_units")}
            arms.append({
                "arm": arm, "selected_ids": selected_ids, "context": [history[i] for i in selected_ids],
                "answer": answer, "budget": details["budget"], "cost_breakdown": parts, **metrics,
            })
        ranking = m2_rank_tfidf(view, resolved=True)
        ranks = {message_id: index for index, message_id in enumerate(ranking["id"], 1)}
        examples.append({
            "seed": seed, "scenario": episode["scenario"], "t": t, "question": view["question"],
            "end": probe["end"], "qtype": probe["qtype"], "state": probe["state"],
            "public_history": view["history"], "gold": probe["gold"], "evidence": evidence,
            "old_values": probe["old_values"], "latest_fact_event": fact_event,
            "uses_alias": any(m2_normalize(text) != text for text in (view["question"], history[evidence]["text"])),
            "resolved_evidence_rank": ranks.get(evidence), "arms": arms,
        })
    return examples


def export_strategy_evidence(episodes, outcomes, run_dir, *, cfg=None, audit_keys=M2_DEVELOPMENT_AUDIT_KEYS):
    """导出 M1 开发数据上的四臂表、汇总、审计和检查。"""
    cfg = PROTOCOL_DRAFT if cfg is None else cfg
    episodes = list(episodes)
    run_dir = Path(run_dir).resolve()
    if run_dir.parent != (OUTPUTS_DIR / "development").resolve() or run_dir.name == "local_setup":
        raise ValueError("选择已初始化、单独命名的 development run。")
    paths = {
        "outcomes": run_dir / "tables" / "development_outcomes.csv",
        "summary": run_dir / "tables" / "strategy_by_query.csv",
        "examples": run_dir / "audit" / "audit_examples.json",
        "review": run_dir / "audit" / "audit_review.csv",
        "checks": run_dir / "audit" / "strategy_checks.json",
    }
    for path in paths.values():
        _writable_output(path)
        if path.exists():
            raise FileExistsError(f"M2 导出拒绝覆盖现有产物 {path}。")

    m2_validate_development_outcomes(episodes, outcomes, cfg)
    if not all(audit_truth(episode)["passed"] for episode in episodes):
        raise ValueError("独立真值审计失败。请先检查 M1 输入。")
    examples = m2_build_audit_examples(episodes, outcomes, cfg, list(audit_keys))
    if ({example["qtype"] for example in examples} != {"recent", "historical"}
            or {(example["scenario"], example["state"]) for example in examples}
            != {("stable", "A"), ("abrupt", "A"), ("abrupt", "B")}
            or not {"repeat", "update"} <= {example["latest_fact_event"] for example in examples}
            or not any(example["uses_alias"] for example in examples)):
        raise ValueError("审计题须覆盖两类问题、三个场景状态、别名、更新与同值复述。")
    summary = outcomes.groupby(["scenario", "state", "qtype", "arm"], sort=True).agg(
        n_questions=("y", "size"), accuracy=("y", "mean"), mean_cost=("cost", "mean"),
        mean_utility=("utility", "mean"), coverage_rate=("coverage", "mean"),
        outdated_rate=("outdated", "mean"), mean_construction_ms=("construction_ms", "mean"),
    ).reset_index()
    review = pd.DataFrame([
        {"seed": example["seed"], "t": example["t"], "m1_reviewer": "", "m2_reviewer": "",
         "verdict": "pending", "m1_notes": "", "m2_notes": ""}
        for example in examples
    ])
    checks = {
        "run_id": run_dir.name,
        "source": {"generator": "M1", "episodes": [
            {"scenario": ep["scenario"], "seed": ep["seed"], "questions": len(ep["probes"])} for ep in episodes
        ]},
        "cost_config": {"budgets": cfg["budgets"], "lambda": cfg["lambda"], "cost_reference": cfg["cost_reference"]},
        "arm_order": list(ARMS),
        "fixture_checks": M2_CHECKS,
        "development_checks": {
            "unique_keys_four_arms_and_matching_metadata": "pass",
            "cost_count_and_utility_consistency": "pass",
            "m1_independent_truth_audit": "pass",
            "audit_coverage_and_reconstruction_match_outcomes": "pass",
        },
        "row_counts": {"outcomes": len(outcomes), "summary": len(summary), "audit_questions": len(examples)},
        "human_review_status": "pending",
    }
    save_csv(paths["outcomes"], outcomes)
    save_csv(paths["summary"], summary)
    save_json(paths["examples"], {"questions": examples})
    save_csv(paths["review"], review)
    save_json(paths["checks"], checks)
    return paths


In [ ]:
# 先在共享路径单元设置新的 development RUN_ID，再显式打开开关。
M2_RUN_DEVELOPMENT = False
if M2_RUN_DEVELOPMENT:
    if MODE != "development" or RUN_ID == "local_setup":
        raise RuntimeError("先选择 MODE='development' 和新的 M2 RUN_ID。")
    m2_cfg = PROTOCOL_DRAFT
    m2_dev_episodes = [
        generate_episode(int(m2_cfg["dev_seed"]) + 100 * scenario_index + index, scenario, m2_cfg)
        for scenario_index, scenario in enumerate(SCENARIOS)
        for index in range(int(m2_cfg["n_dev_per_scenario"]))
    ]
    m2_handoff = next(ep for ep in m2_dev_episodes if ep["scenario"] == "abrupt")
    m2_handoff_frame = evaluate_episode(m2_handoff, m2_cfg)
    m2_validate_development_outcomes([m2_handoff], m2_handoff_frame, m2_cfg)
    m2_handoff_path = RUN_DIR / "tables" / f"m3_handoff_abrupt_{m2_handoff['seed']}.csv"
    save_csv(m2_handoff_path, m2_handoff_frame)
    print("M3 单对话四臂表", m2_handoff_path, len(m2_handoff_frame), "行")

    m2_dev_outcomes = pd.concat([
        m2_handoff_frame if ep is m2_handoff else evaluate_episode(ep, m2_cfg)
        for ep in m2_dev_episodes
    ], ignore_index=True)
    m2_exported = export_strategy_evidence(m2_dev_episodes, m2_dev_outcomes, RUN_DIR, cfg=m2_cfg)
    print("M2 开发产物", RUN_DIR, len(m2_dev_outcomes), "行")


### M2 开发联调与审计

- 手工 fixture 的 `t=3` 中，短窗选中了最新语言事实，但少了 focus，回答为 UNKNOWN。这说明最新证据覆盖与答对是不同指标。
- 开发案例 `seed=130100, t=99` 中，`resolved_tfidf` 保留了 focus，最新证据 `id=323` 在归一化排序中排第 9，未进入前 8 条候选，回答为旧值。本轮保留当前策略，供后续评估排序与候选上限。
- 导出开关默认关闭。启用时先生成 M3 单对话表，再保存全量开发表、按题型汇总及十题审计。人工复核由 M1/M2 逐题填写。

复跑、比较和交接记录见 [M2 研究记录](deliverables/M2_notes.md)。


<a id="member-m3"></a>
## M3 · 贝叶斯建模与回放
**后验更新、决策、优化、回放与数值验证由你实现。任务区间m3-*。**
输入：问题、已知四候选cost、M2单对话评价表；输出：统一学习器、回放、推导与核查；M4接收。

1. **先推导。** 从Bernoulli似然/Beta先验推导TS后验，手算收到1、1、0后的参数与均值；解释成本为什么不应计为失败次数。
2. **实现TS与参照。** 定义状态、采样和成本决策；实现四固定臂与基于公开问题的RuleRouter，研究先验和探索。
3. **研究CTS。** 每臂贝叶斯logistic与高斯先验；自己推导MAP目标、梯度、Hessian、Laplace协方差。实现优化、采样、每5次已选反馈重拟合；解释特征共线性与数值稳定性。
4. **实现回放。** 每新对话新建学习器；choose只看q/costs，update只看已选臂0/1反馈；全臂表仅属于评估器。
5. **验证交付。** 自己写手算一致、有限差分、协方差、重置、反馈隔离案例；报告拟合失败，不静默吞掉。交M4微型输入/输出及M2真实表上的检查。

**研究问题：** 如何检测未选奖励泄漏？同seed是否复现？特征如何不借用隐藏标签？TS与CTS差异是否足以构成严格特征消融？
**输出：** audit/method_checks.json、tables/development_replay.csv；推导和局限写M3_notes。


In [ ]:
import re

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit


def question_features(question):
    """Intercept, explicit project ID, and the pronoun 'its'."""
    return np.array([
        1.0,
        float(bool(re.search(r"\bP[0-9]+\b", question, re.IGNORECASE))),
        float(bool(re.search(r"\bits\b", question, re.IGNORECASE))),
    ])


def _m3_objective(weights, features, rewards, precision):
    logits = features @ weights
    loss = np.logaddexp(0, logits).sum() - rewards @ logits
    gradient = features.T @ (expit(logits) - rewards) + precision * weights
    return loss + 0.5 * precision * (weights @ weights), gradient


def _m3_hessian(weights, features, precision):
    probabilities = expit(features @ weights)
    curvature = probabilities * (1 - probabilities)
    return precision * np.eye(len(weights)) + features.T @ (curvature[:, None] * features)


class Selector:
    def __init__(self, method, seed, cfg):
        if method not in METHODS:
            raise ValueError(f"Unknown method: {method}")
        self.method = method
        self.rng = np.random.default_rng(seed)
        self.cost_weight = cfg["lambda"] / cfg["cost_reference"]
        self.precision = cfg["prior_precision"]
        self.refit_every = cfg["refit_every"]
        self.successes = np.zeros(len(ARMS), dtype=int)
        self.failures = np.zeros(len(ARMS), dtype=int)
        self.features = [[] for _ in ARMS]
        self.rewards = [[] for _ in ARMS]
        self.mean = np.zeros((len(ARMS), 3))
        self.covariance = np.repeat((np.eye(3) / self.precision)[None], len(ARMS), axis=0)

    def choose(self, question, costs):
        costs = np.asarray(costs, dtype=float)
        if costs.shape != (len(ARMS),) or not np.isfinite(costs).all() or (costs < 0).any():
            raise ValueError("Expected one finite, nonnegative cost per arm in ARMS order")
        if self.method in ARMS:
            return ARMS.index(self.method)
        if self.method == "TS":
            probabilities = self.rng.beta(1 + self.successes, 1 + self.failures)
        else:
            features = question_features(question)
            if self.method == "RuleRouter":
                arm = "recent_short" if features[2] and not features[1] else "resolved_tfidf"
                return ARMS.index(arm)
            weights = np.array([
                self.rng.multivariate_normal(mean, covariance)
                for mean, covariance in zip(self.mean, self.covariance)
            ])
            probabilities = expit(weights @ features)
        return int(np.argmax(probabilities - self.cost_weight * costs))

    def update(self, arm_index, reward, question):
        if not 0 <= arm_index < len(ARMS) or reward not in (0, 1):
            raise ValueError("Expected a valid arm index and a binary reward")
        if self.method == "TS":
            self.successes[arm_index] += reward
            self.failures[arm_index] += 1 - reward
        if self.method != "LaplaceCTS":
            return

        features = self.features[arm_index]
        rewards = self.rewards[arm_index]
        feature = question_features(question)
        if (len(rewards) + 1) % self.refit_every == 0:
            design = np.array([*features, feature])
            observed = np.array([*rewards, reward])
            fit = minimize(
                _m3_objective, self.mean[arm_index],
                args=(design, observed, self.precision), jac=True,
                method="BFGS", options={"gtol": 1e-6, "maxiter": 100},
            )
            gradient_norm = np.linalg.norm(fit.jac)
            if not np.isfinite(fit.x).all() or not np.isfinite(gradient_norm) or gradient_norm > 1e-3:
                raise RuntimeError(f"CTS MAP fit failed for {ARMS[arm_index]}: {fit.message}")
            hessian = _m3_hessian(fit.x, design, self.precision)
            covariance = np.linalg.solve(hessian, np.eye(3))
            self.mean[arm_index] = fit.x
            self.covariance[arm_index] = (covariance + covariance.T) / 2
        features.append(feature)
        rewards.append(reward)


def replay_episode(arm_outcomes, method, algorithm_seed, cfg):
    """Replay one dialogue in chronological order with chosen-arm feedback."""
    selector = Selector(method, algorithm_seed, cfg)
    outcome_columns = ["question", "y", "cost", "coverage", "outdated"]
    required = ["seed", "scenario", "t", "arm", *outcome_columns]
    missing = set(required) - set(arm_outcomes.columns)
    if missing:
        raise ValueError(f"Missing outcome columns: {sorted(missing)}")
    frame = arm_outcomes[required]
    if frame.empty or frame.isna().any().any() or len(frame[["seed", "scenario"]].drop_duplicates()) != 1:
        raise ValueError("Expected a nonempty, complete table for one dialogue")
    if not frame.y.isin([0, 1]).all():
        raise ValueError("Expected binary correctness")
    seed, scenario = frame[["seed", "scenario"]].iloc[0]
    rows = []
    for t, group in frame.groupby("t", sort=True):
        if len(group) != len(ARMS) or set(group.arm) != set(ARMS) or group.question.nunique() != 1:
            raise ValueError(f"Expected one row per arm and one shared question at t={t}")
        group = group.set_index("arm").loc[list(ARMS)]
        question = group.question.iloc[0]
        arm_index = selector.choose(question, group.cost.to_numpy(dtype=float))
        arm = ARMS[arm_index]
        outcome = group.loc[arm]
        reward = int(outcome.y)
        selector.update(arm_index, reward, question)
        rows.append(dict(
            seed=int(seed), scenario=scenario, t=int(t), method=method,
            alg_seed=algorithm_seed, arm=arm, y=reward, cost=outcome.cost,
            utility=reward - cfg["lambda"] * outcome.cost / cfg["cost_reference"],
            coverage=int(outcome.coverage), outdated=int(outcome.outdated),
        ))
    return pd.DataFrame(rows)


In [ ]:
def run_m3_method_checks(arm_outcomes, cfg):
    """Return audit measurements and replay rows after checking the development dialogue."""
    question = "What is its current dataset?"
    ts = Selector("TS", 17, cfg)
    for reward in (1, 1, 0):
        ts.update(0, reward, question)
    assert (1 + ts.successes[0], 1 + ts.failures[0]) == (3, 2)
    assert not (ts.successes[1:] + ts.failures[1:]).any()
    np.testing.assert_array_equal(question_features(question), [1, 0, 1])
    np.testing.assert_array_equal(question_features("What is P1's metric?"), [1, 1, 0])

    cts = Selector("LaplaceCTS", 17, cfg)
    prior = cts.covariance.copy()
    interval = cfg["refit_every"]
    questions = [question, "What is P1's metric?"]
    for i in range(interval - 1):
        cts.update(1, int(i % 2 == 0), questions[i % 2])
    np.testing.assert_array_equal(cts.mean, 0)
    np.testing.assert_array_equal(cts.covariance, prior)
    cts.update(1, 1, question)
    features = np.array(cts.features[1])
    rewards = np.array(cts.rewards[1])
    assert [len(history) for history in cts.rewards] == [0, interval, 0, 0]
    np.testing.assert_array_equal(cts.mean[[0, 2, 3]], 0)
    np.testing.assert_array_equal(cts.covariance[[0, 2, 3]], prior[[0, 2, 3]])
    np.testing.assert_array_equal(Selector("LaplaceCTS", 17, cfg).covariance, prior)

    precision = cfg["prior_precision"]
    weights = cts.mean[1]
    point = weights + np.array([0.2, -0.1, 0.3])
    step = 1e-5
    identity = np.eye(3)
    plus = [_m3_objective(point + step * d, features, rewards, precision) for d in identity]
    minus = [_m3_objective(point - step * d, features, rewards, precision) for d in identity]
    fd_gradient = np.array([(a[0] - b[0]) / (2 * step) for a, b in zip(plus, minus)])
    fd_hessian = np.column_stack([(a[1] - b[1]) / (2 * step) for a, b in zip(plus, minus)])
    analytic_gradient = _m3_objective(point, features, rewards, precision)[1]
    analytic_hessian = _m3_hessian(point, features, precision)
    np.testing.assert_allclose(fd_gradient, analytic_gradient, atol=1e-7)
    np.testing.assert_allclose(fd_hessian, analytic_hessian, atol=1e-7)
    map_gradient_norm = np.linalg.norm(_m3_objective(weights, features, rewards, precision)[1])
    assert map_gradient_norm < 1e-3
    covariance = cts.covariance[1]
    np.testing.assert_allclose(_m3_hessian(weights, features, precision) @ covariance, identity, atol=1e-8)
    min_eigenvalue = np.linalg.eigvalsh(covariance).min()
    assert min_eigenvalue > 0

    results = []
    for method in METHODS:
        for seed in cfg["algorithm_seeds"]:
            replay = replay_episode(arm_outcomes, method, seed, cfg)
            shuffled = arm_outcomes.sample(frac=1, random_state=seed)
            pd.testing.assert_frame_equal(replay, replay_episode(shuffled, method, seed, cfg))
            selected = pd.MultiIndex.from_frame(replay[["t", "arm"]])
            unused = ~pd.MultiIndex.from_frame(arm_outcomes[["t", "arm"]]).isin(selected)
            modified = arm_outcomes.copy()
            modified.loc[unused, "y"] = 1 - modified.loc[unused, "y"]
            modified["utility"] = modified.y - cfg["lambda"] * modified.cost / cfg["cost_reference"]
            pd.testing.assert_frame_equal(replay, replay_episode(modified, method, seed, cfg))
            results.append(replay)
    table = pd.concat(results, ignore_index=True)
    return {
        "beta_posterior": [3, 2],
        "map_gradient_norm": float(map_gradient_norm),
        "gradient_fd_max_error": float(np.max(np.abs(fd_gradient - analytic_gradient))),
        "hessian_fd_max_error": float(np.max(np.abs(fd_hessian - analytic_hessian))),
        "covariance_min_eigenvalue": float(min_eigenvalue),
        "refit_every_selected_feedbacks": interval,
        "reset_and_unchosen_reward_flip": "passed",
        "methods": list(METHODS),
        "algorithm_seeds": list(cfg["algorithm_seeds"]),
        "replay_rows": len(table),
    }, table


def export_m3_development(handoff_path, run_id, cfg):
    """Run §0 first, then call this with a local M2 development CSV and a new run ID."""
    import hashlib

    handoff_path = Path(handoff_path)
    outcomes = pd.read_csv(handoff_path)
    checks, table = run_m3_method_checks(outcomes, cfg)
    checks["source"] = {
        "file": handoff_path.name,
        "sha256": hashlib.sha256(handoff_path.read_bytes()).hexdigest(),
        "seed": int(outcomes.seed.iloc[0]),
        "scenario": outcomes.scenario.iloc[0],
        "rows": len(outcomes),
    }
    run = init_run("development", run_id)
    save_json(run / "audit" / "method_checks.json", checks)
    save_csv(run / "tables" / "development_replay.csv", table)
    return run


### M3 · 开发核查与交接
先运行第0节、共同协议与本节两个代码单元。将M2单对话CSV保存到本地后运行：

```python
export_m3_development("team_workspace/m2_handoff/m3_handoff_abrupt_130100.csv",
                      "m3_abrupt_130100", PROTOCOL_DRAFT)
```

M4调用 `replay_episode(outcomes, method, algorithm_seed, cfg)`。每次调用重置学习器。
推导、来源与验收范围见 [M3交接记录](deliverables/M3_notes.md)。


<a id="member-m4"></a>
## M4 · 集成、实验设计与统计
**你要编写调度、开发选择、冻结、统计和归档。任务区间m4-*。**
输入：M1–M3实现、证据、审查；输出：真实run、表、manifest和判断；M3复核、M5接收。

1. **先做统计手算。** 自建少量对话小表验证配对和算法seed的对话内平均；这是fixture，不能作为研究结果。
2. **接通demo。** 编写入口逐模块调用；按场景仅用开发平均效用选固定基线，说明平局规则，不用测试结果选基线。
3. **实现冻结与失败管理。** 核验M1/M2审计、M3方法检查、全组确认；保存配置/源码/证据快照、哈希、版本，检查seed分离、防覆盖；登记每次正式尝试和已看数据情况。
4. **正式调度。** 每场景30条×300题，四臂评价与方法回放；新对话重置，两个算法seed不是两份独立数据。先查完整性再看效果。
5. **自行实现统计与判断。** 对话级配对bootstrap，主次比较、95%及两主比较97.5%区间；区间含0不等于等效，点估计到0.02不等于下界超过0.02。
6. **可复现交付。** 导出同run完整表和manifest，让M3复核聚合/配对/区间，M5独立读取。保留负结果，不换seed直到显著。

**输出：** data/evaluation_dialogues.jsonl.gz；tables/evaluation_outcomes.csv、replay.csv、episode_metrics.csv、primary_effects.csv、secondary_effects.csv、method_summary.csv；config/frozen_config.json、notebook_snapshot.ipynb；audit/review_snapshot.json；reports/M4_handoff.md；根级manifest.json。实现与产物均待你完成。


In [ ]:
import numpy as np
import pandas as pd

M4_METRICS = ['y', 'cost', 'utility', 'coverage', 'outdated']
M4_EFFECT_COLUMNS = ['scenario', 'method', 'baseline', 'n', 'effect',
                     'low', 'high', 'family_low', 'family_high']


def _m4_validate(frame, keys, metrics):
    required = set(keys + metrics)
    if frame.empty or not required.issubset(frame.columns):
        raise ValueError('Empty input or missing columns')
    if frame[list(required)].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError('Null values or duplicate keys')
    if not np.isfinite(frame[metrics].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite metrics')


def choose_fixed_baselines(development_outcomes):
    """Development only: equal dialogue weights; exact ties use ARMS order.

    Caller must verify development provenance before calling; a table alone
    cannot prove that its observations were not taken from evaluation data.
    Returns {scenario: arm}.
    """
    f = development_outcomes
    _m4_validate(f, ['scenario', 'seed', 't', 'arm'], ['utility'])
    for _, group in f.groupby(['scenario', 'seed', 't']):
        if set(group.arm) != set(ARMS):
            raise ValueError('Incomplete four-arm development table')
    means = f.groupby(['scenario', 'seed', 'arm']).utility.mean()
    scores = means.groupby(['scenario', 'arm']).mean().unstack().reindex(columns=ARMS)
    return {scenario: row.idxmax() for scenario, row in scores.iterrows()}


def aggregate_dialogues(replay_table):
    """Average questions within trajectory, then algorithm seeds per dialogue.

    Validate observed method/seed grids. The scheduler must additionally check
    these against frozen expected methods, algorithm seeds and T: completely
    absent trajectories cannot be inferred from an observed table alone.
    """
    f = replay_table
    keys = ['scenario', 'seed', 't', 'method', 'alg_seed']
    _m4_validate(f, keys, M4_METRICS)
    if not f[['y', 'coverage', 'outdated']].isin([0, 1]).all().all() or (f.cost < 0).any():
        raise ValueError('Invalid binary metric or negative cost')
    if not np.equal(f.t, np.floor(f.t)).all() or (f.t < 0).any():
        raise ValueError('Invalid question index')
    methods = set(f.method)
    seed_sets = {m: set(g.alg_seed) for m, g in f.groupby('method')}
    for _, episode in f.groupby(['scenario', 'seed']):
        if set(episode.method) != methods:
            raise ValueError('Missing method in dialogue')
        expected_t = set(range(int(episode.t.max()) + 1))
        for method, group in episode.groupby('method'):
            if set(group.alg_seed) != seed_sets[method]:
                raise ValueError('Missing algorithm seed')
            for _, trajectory in group.groupby('alg_seed'):
                if set(trajectory.t) != expected_t:
                    raise ValueError('Incomplete question trajectory')
    per_seed = f.groupby(['scenario', 'seed', 'method', 'alg_seed'])[M4_METRICS].mean()
    return per_seed.groupby(['scenario', 'seed', 'method']).mean().reset_index()


def paired_effects(episode_metrics, baselines, cfg):
    """Return (primary, secondary): CTS primary, TS exploratory, utility effect.

    Percentile bootstrap resamples paired dialogue differences. Primary
    family intervals use 97.5% coverage for the two prespecified scenarios.
    """
    f = episode_metrics
    _m4_validate(f, ['scenario', 'seed', 'method'], M4_METRICS)
    if set(f.scenario) != set(SCENARIOS) or set(baselines) != set(SCENARIOS):
        raise ValueError('Both prespecified scenarios and baselines required')
    count = cfg['n_bootstrap']
    if isinstance(count, bool) or not isinstance(count, (int, np.integer)) or count < 1:
        raise ValueError('n_bootstrap must be a positive integer')
    rng = np.random.default_rng(cfg['bootstrap_seed'])
    primary, secondary = [], []
    for scenario in SCENARIOS:
        baseline = baselines[scenario]
        if baseline not in ARMS:
            raise ValueError('Baseline must be a fixed arm')
        wide = f[f.scenario == scenario].pivot(index='seed', columns='method', values='utility').sort_index()
        needed = [baseline, 'LaplaceCTS', 'TS']
        if not set(needed).issubset(wide.columns) or wide[needed].isna().any().any():
            raise ValueError('Incomplete paired dialogues')
        if len(wide) < 2:
            raise ValueError('At least two independent dialogues required')
        for method, target in [('LaplaceCTS', primary), ('TS', secondary)]:
            differences = (wide[method] - wide[baseline]).to_numpy()
            samples = np.empty(count)
            for i in range(count):
                samples[i] = rng.choice(differences, size=len(differences), replace=True).mean()
            low, high = np.quantile(samples, [.025, .975])
            family = np.quantile(samples, [.0125, .9875]) if method == 'LaplaceCTS' else [np.nan, np.nan]
            target.append(dict(scenario=scenario, method=method, baseline=baseline,
                               n=len(differences), effect=differences.mean(), low=low,
                               high=high, family_low=family[0], family_high=family[1]))
    return (pd.DataFrame(primary, columns=M4_EFFECT_COLUMNS),
            pd.DataFrame(secondary, columns=M4_EFFECT_COLUMNS))


def freeze_protocol(cfg, baselines, evidence_paths, run_dir):
    """Delegate to M4's approval-bound protocol implementation.

    cfg is a complete plan wrapper (config/evaluation_seeds/used_seeds/baselines).
    Keep module import local for notebook execution from the repository root.
    """
    from scripts.m4_protocol import freeze
    from pathlib import Path
    if cfg.get('baselines') != baselines:
        raise ValueError('Baseline differs from approval plan')
    root = Path.cwd()
    code_files = dict(notebook=root/'ARIN7101_Context_Bandit_Release_Notebook.ipynb',
        runner=root/'scripts/run_m4_formal.py', protocol=root/'scripts/m4_protocol.py',
        loader=root/'scripts/test_member_handoff.py', analysis=root/'scripts/m4_analysis.py')
    return freeze(cfg, evidence_paths, code_files, run_dir)


def run_evaluation(episodes, methods, algorithm_seeds, cfg):
    """Integrate M1/M2/M3 with exact configured grids, no silent dropped rows."""
    episodes = list(episodes)
    if not episodes or len(methods) != len(set(methods)) or set(methods) != set(METHODS):
        raise ValueError('Expected all seven methods exactly once')
    if list(algorithm_seeds) != list(cfg['algorithm_seeds']) or len(set(algorithm_seeds)) != len(algorithm_seeds):
        raise ValueError('Algorithm seeds differ from configuration')
    if len({e['seed'] for e in episodes}) != len(episodes):
        raise ValueError('Dialogue seed collision')
    all_outcomes, all_replay = [], []
    for episode in episodes:
        if episode['scenario'] not in SCENARIOS:
            raise ValueError('Unexpected scenario')
        if [p['t'] for p in episode['probes']] != list(range(cfg['T'])):
            raise ValueError('Dialogue question count/order differs from T')
        outcomes = evaluate_episode(episode, cfg)
        expected = {(t, a) for t in range(cfg['T']) for a in ARMS}
        if len(outcomes) != len(expected) or set(zip(outcomes.t, outcomes.arm)) != expected:
            raise ValueError('Incomplete/duplicate four-arm grid')
        if set(outcomes.seed) != {episode['seed']} or set(outcomes.scenario) != {episode['scenario']}:
            raise ValueError('Outcome identity mismatch')
        if not np.allclose(outcomes.utility, outcomes.y-cfg['lambda']*outcomes.cost/cfg['cost_reference']):
            raise ValueError('Inconsistent utility')
        all_outcomes.append(outcomes)
        for method in methods:
            for algorithm_seed in algorithm_seeds:
                replay = replay_episode(outcomes, method, algorithm_seed, cfg)
                if len(replay) != cfg['T'] or set(replay.t) != set(range(cfg['T'])):
                    raise ValueError('Incomplete replay trajectory')
                for field, value in [('seed', episode['seed']), ('scenario', episode['scenario']),
                                     ('method', method), ('alg_seed', algorithm_seed)]:
                    if set(replay[field]) != {value}:
                        raise ValueError('Replay identity mismatch: '+field)
                joined = replay.merge(outcomes, on=['seed','scenario','t','arm'], suffixes=('', '_expected'), validate='many_to_one')
                if len(joined) != len(replay):
                    raise ValueError('Unknown selected arm')
                for metric in M4_METRICS:
                    if not np.allclose(joined[metric], joined[metric+'_expected']):
                        raise ValueError('Selected outcome mismatch: '+metric)
                all_replay.append(replay)
    result = pd.concat(all_replay, ignore_index=True)
    aggregate_dialogues(result)  # binary/finite/grid checks before effect inspection
    return pd.concat(all_outcomes, ignore_index=True), result


def export_run(run_dir, artifacts, metadata):
    """Archive a development/demo run only; hashes exclude manifest itself."""
    import hashlib
    import json
    from pathlib import Path
    root = Path(run_dir).resolve()
    if metadata.get('mode') == 'formal':
        from scripts.m4_protocol import verify_frozen
        frozen = verify_frozen(metadata['frozen_directory'])
        if frozen['plan_sha256'] != metadata['plan_sha256'] or frozen['plan']['config'] != metadata['config']:
            raise ValueError('Archive does not match frozen plan')
    if metadata.get('mode') not in ('development', 'demo', 'formal'):
        raise ValueError('Formal archive requires the future formal lifecycle')
    if (root / 'manifest.json').exists():
        raise FileExistsError('Manifest already exists')
    required = {'tables/evaluation_outcomes.csv', 'tables/replay.csv',
                'tables/episode_metrics.csv', 'tables/primary_effects.csv',
                'tables/secondary_effects.csv', 'tables/method_summary.csv'}
    if not required.issubset(artifacts):
        raise ValueError('Missing result tables')
    if metadata['mode'] in ('demo', 'formal'):
        cfg = metadata['config']
        seeds = metadata['seeds']['evaluation']
        formal = metadata['mode'] == 'formal'
        per_scenario = cfg['n_formal_per_scenario' if formal else 'n_demo_per_scenario']
        if len(set(seeds)) != len(SCENARIOS) * per_scenario or len(seeds) != len(set(seeds)):
            raise ValueError('Evaluation seed count differs from design')
        outcomes = artifacts['tables/evaluation_outcomes.csv']
        identities = outcomes[['scenario', 'seed']].drop_duplicates()
        if (set(identities.seed) != set(seeds) or identities.seed.duplicated().any()
                or identities.groupby('scenario').size().to_dict() != dict.fromkeys(SCENARIOS, per_scenario)):
            raise ValueError('Evaluation dialogues differ from design')
        if formal and set(identities.itertuples(index=False, name=None)) != {
                (sc, seed) for sc, group in frozen['plan']['evaluation_seeds'].items() for seed in group}:
            raise ValueError('Dialogue identities differ from frozen plan')
        # Distinct valid keys plus the full product size prove complete coverage.
        grids = [
            ('evaluation_outcomes', dict(t=range(cfg['T']), arm=ARMS)),
            ('replay', dict(t=range(cfg['T']), method=METHODS, alg_seed=cfg['algorithm_seeds'])),
            ('episode_metrics', dict(method=METHODS)),
        ]
        for table, axes in grids:
            frame = artifacts['tables/' + table + '.csv']
            keys = ['seed', *axes]
            expected_size = len(seeds)
            for column, values in axes.items():
                expected_size *= len(values)
                if not frame[column].isin(values).all():
                    raise ValueError('Unexpected ' + column + ' in ' + table)
            if (len(frame) != expected_size or frame.duplicated(keys).any()
                    or not frame['scenario'].eq(frame.seed.map(identities.set_index('seed').scenario)).all()):
                raise ValueError('Incomplete or inconsistent export grid: ' + table)
    targets = {}
    for name, value in artifacts.items():
        path = (root / name).resolve()
        if not path.is_relative_to(root) or path == root or path.exists() or name == 'manifest.json':
            raise ValueError('Unsafe or existing artifact: '+name)
        targets[name] = path
    for name, path in targets.items():
        path.parent.mkdir(parents=True, exist_ok=True)
        value = artifacts[name]
        if isinstance(value, pd.DataFrame):
            value.to_csv(path, index=False, encoding='utf-8', mode='x')
        else:
            with path.open('xb') as stream:
                stream.write(value if isinstance(value, bytes) else value.encode('utf-8'))
    files = {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in sorted(root.rglob('*')) if p.is_file()}
    manifest = dict(metadata, status=('computed_pending_review' if metadata['mode']=='formal' else 'demo_complete_pending_review'), files=files)
    with (root / 'manifest.json').open('x', encoding='utf-8') as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2, allow_nan=False)
    return manifest


In [ ]:
# Synthetic fixture only; no research data, output files or review signatures.
def m4_fixture_checks():
    rows = []
    for scenario in SCENARIOS:
        for seed in [1, 2]:
            for method in [ARMS[0], 'LaplaceCTS', 'TS']:
                for alg_seed in [17, 37]:
                    for t in [0, 1]:
                        # Baseline 0; CTS dialogue effects .25 and .75; TS zero.
                        y = int(method == 'LaplaceCTS' and
                                ((seed == 1 and alg_seed == 17 and t == 0) or
                                 (seed == 2 and not (alg_seed == 17 and t == 0))))
                        rows.append(dict(scenario=scenario, seed=seed, method=method,
                                         alg_seed=alg_seed, t=t, y=y, cost=0,
                                         utility=float(y), coverage=y, outdated=0))
    fixture = pd.DataFrame(rows)
    metrics = aggregate_dialogues(fixture)
    assert len(metrics) == 12
    assert metrics.query("method == 'LaplaceCTS'").utility.tolist() == [.25, .75, .25, .75]
    baselines = dict.fromkeys(SCENARIOS, ARMS[0])
    config = dict(n_bootstrap=2000, bootstrap_seed=2026)
    primary, secondary = paired_effects(metrics, baselines, config)
    assert (primary.n == 2).all() and (primary.effect == .5).all()
    # Exact bootstrap distribution for two differences: .25/.5/.75,
    # with probabilities 1/4, 1/2, 1/4. Both interval endpoints are .25/.75.
    assert (primary.low == .25).all() and (primary.high == .75).all()
    assert (primary.family_low == .25).all() and (primary.family_high == .75).all()
    assert (secondary[['effect', 'low', 'high']] == 0).all().all()
    pd.testing.assert_frame_equal(primary, paired_effects(metrics.sample(frac=1, random_state=9), baselines, config)[0])
    negative = metrics.copy()
    negative.loc[negative.method == ARMS[0], 'utility'] = 1.
    assert (paired_effects(negative, baselines, config)[0].effect == -.5).all()
    dev = pd.DataFrame([dict(scenario=s, seed=d, t=0, arm=a, utility=0.)
                        for s in SCENARIOS for d in [1, 2] for a in ARMS])
    assert choose_fixed_baselines(dev) == baselines  # deterministic exact tie
    dev.loc[dev.arm == ARMS[2], 'utility'] = .2
    assert set(choose_fixed_baselines(dev).values()) == {ARMS[2]}
    def rejects(fn, *args):
        try:
            fn(*args)
        except ValueError:
            return
        raise AssertionError('Malformed fixture was accepted')
    rejects(aggregate_dialogues, pd.concat([fixture, fixture.iloc[[0]]]))
    rejects(aggregate_dialogues, fixture.drop(index=0))
    rejects(aggregate_dialogues, fixture[~((fixture.seed == 1) & (fixture.method == 'TS') & (fixture.alg_seed == 37))])
    rejects(aggregate_dialogues, fixture[~((fixture.seed == 1) & (fixture.method == 'TS'))])
    rejects(aggregate_dialogues, fixture.assign(utility=np.nan))
    rejects(paired_effects, metrics.drop(index=0), baselines, config)
    rejects(choose_fixed_baselines, dev.drop(index=0))
    print('M4 synthetic fixture checks passed; n=2 per scenario, CTS effect=0.5, CI=[0.25,0.75]. Not research results.')

m4_fixture_checks()


In [ ]:
# Experimental-design exploration: synthetic fixtures only.
def m4_design_exploration():
    rng = np.random.default_rng(20261119)
    repeats = 10000
    dialogue_differences = np.array([0., 1.])
    paired = rng.choice(dialogue_differences, size=(repeats, 2)).mean(axis=1)
    question_differences = np.repeat(dialogue_differences, 100)
    incorrect = rng.choice(question_differences, size=(repeats, 200)).mean(axis=1)
    paired_ci = np.quantile(paired, [.025, .975])
    incorrect_ci = np.quantile(incorrect, [.025, .975])
    baseline = np.array([-.2, .6])
    treatment = baseline + .1
    indices = rng.integers(0, 2, size=(repeats, 2))
    paired_effect = (treatment[indices] - baseline[indices]).mean(axis=1)
    independent_effect = (treatment[rng.integers(0, 2, size=(repeats, 2))].mean(axis=1)
                          - baseline[rng.integers(0, 2, size=(repeats, 2))].mean(axis=1))
    assert np.allclose(paired_ci, [0, 1])
    assert np.ptp(incorrect_ci) < np.ptp(paired_ci)
    assert np.allclose(paired_effect, .1)
    assert independent_effect.std() > paired_effect.std()
    result = pd.DataFrame([
        ['dialogue bootstrap (correct unit)', *paired_ci],
        ['question bootstrap (incorrect unit)', *incorrect_ci],
        ['paired constant-effect fixture', *np.quantile(paired_effect, [.025, .975])],
        ['independent resampling loses pairing', *np.quantile(independent_effect, [.025, .975])],
    ], columns=['synthetic_design', 'low', 'high'])
    print('FIXTURE ONLY: not project effects, power estimates or coverage guarantees.')
    print(result.to_string(index=False))
    return result

M4_DESIGN_FIXTURE = m4_design_exploration()


### M4结果判断（实际实验后本人填写）
主比较、基线选择依据、统计单位和n；效应方向/区间/0.02含义；负结果、异常、修正与已看情况；M3复核；交付run/commit/manifest校验和：待填写。未完成前只写计划。


<a id="member-m5"></a>
## M5 · 可视化研究、结果解释与汇报
**图表代码、读取校验与评价由你完成。任务区间m5-*。**
输入：M4完整归档及M1–M3说明；输出：自己设计的图、解读与汇报；全组复核。

1. **不等结果开始。** 设计叙事与每图的问题，用明确标为fixture的小表探索图型/颜色/误差线。最终不能混入假数据。
2. **实现独立读取。** 指定run路径，校验manifest、文件、模式、配置和规模，不依赖M4内存，不自动取“最新目录”；和M4共同验收契约。
3. **自主设计与编码。** 下表是问题菜单，不是画法答案。解释图型、聚合和区间选择；可协商合并或调整图数，不为凑数重复展示。
4. **自己评估。** 对照CSV复核数值，每图写观察→证据→局限，与对应成员核对。不显著或方向相反也如实展示，不能复制参考本结论。
5. **完成汇报。** M1背景、M2策略、M3方法、M4统计、M5叙事；旧计划约15–16页slides/15–20分钟视频，须核对课程要求，说明个人贡献与AI辅助。

| 建议问题 | 需要的数据 | 联合复核 |
|---|---|---|
| 主效应是否明确？ | primary/secondary_effects | M4/M3 |
| 准确率、成本、效用如何取舍？ | episode_metrics/method_summary | M2 |
| 效用差由什么组成？ | 同一基线的分量差 | M2/M4 |
| 不同需求怎样选臂？ | replay及题型 | M3/M1 |
| 收益是否贯穿对话？ | 配对逐题轨迹 | M4 |
| 策略有何互补/失败机制？ | 全臂表与具体例子 | M1/M2 |
| 突变前后是否相同？ | 阶段对话配对表 | M4 |

**边界：** 描述性区间不增加主检验；逐点带不是同时置信带；固定基线累计差不是理论regret；proxy成本不是实际费用。
**输出：** figures/<figure_id>.png/.svg、tables/figure_<figure_id>.csv、reports/figure_interpretations.md、reports/figure_manifest.json。只新增图表文件，不改M4数据；重绘使用新figure_id或明确render子目录并记录来源。


In [ ]:
def load_verified_run(run_dir):
    """M5实现：独立校验归档后读取绘图所需数据。"""
    raise NotImplementedError("M5：实现来源校验和读取")

def build_figures(results, cfg):
    """M5实现：自行选择图型、聚合/误差线，返回图对象和对应数据表。"""
    raise NotImplementedError("M5：设计并实现可视化")

def export_visual_evidence(figures, figure_tables, source_run, render_id):
    """M5实现：保存图/表/来源清单，不修改上游原始数据。"""
    raise NotImplementedError("M5：实现图表交付")


In [ ]:
# TODO M5：用fixture先检查绘图；模拟缺文件/改CSV/混run验证读取器。
# 新内核只执行共享单元与M5模块，不依赖M4变量。
# 检查数据-图一致、单位/区间、负值标签、图例。
# 实现后增加绘图调用；本人写解读，不用固定文字自动宣布胜出。


### M5逐图研究记录（本人填写）
每图的问题、来源run与表、汇总方式、观察、支持/不支持的结论、局限、复核人：待填写。
随后补slides、分工、视频链接与提交清单。未有结果时只写计划与示意图设计。


## 完成标准
todo → doing → review → done：自己的实现与研究说明完成，自己设计的案例通过，接收人能按接口使用并复核后再标done。结构检查不证明算法正确或项目完成。

正式评估前冻结共同协议；看过测试结果之后的探索、修正和失败必须留记录，不能反复换seed至显著。

课程线索：Lecture 2的Beta/MAP/Laplace，Lecture 6的bandit/后验采样，Lecture 4的开发/评估隔离，Lecture 1的建模与不确定性。成员自行阅读原课件、推导和引用，本工作本不提供完成的推导答案。

模块完成后由M4自行编写集成入口；各成员用自己的测试、实验与交叉复核证明成果。
